## Language Modeling Tutorial

Prepared by Ye Kyaw Thu, Language Understanding Lab., Myanmar  
For AI Fundamental Class  
Last updated: 2 May 2026  


In [30]:
%cd /mnt/disk1/ye/exp/LM-Tutorial

/mnt/disk1/ye/exp/LM-Tutorial


In [32]:
!ls

bk	    general.log  lm_toy.test.txt   lm_toy.val.txt  transformer
data	    general.txt  lm_toy.train.txt  lstm
domain.log  kenlm	 lm_toy.txt	   tmp_kenlm


In [33]:
import torch
import os

# Set working directory to your fast NVMe drive
WORK_DIR = "/mnt/disk1/ye/exp/LM-Tutorial/"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"Working Directory: {os.getcwd()}")

# Verify GPU
print("\n--- GPU Status ---")
if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Detected: {gpu_name}")
    print(f"VRAM: {gpu_mem:.2f} GB")
    # Enable TF32 for faster matrix math on Ampere architecture (RTX 3090)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    device = torch.device("cpu")
    print("WARNING: CUDA not found. Falling back to CPU.")

# Verify CPU cores
import multiprocessing
print(f"\n--- CPU Status ---")
print(f"CPU Cores available: {multiprocessing.cpu_count()}")

Working Directory: /mnt/disk1/ye/exp/LM-Tutorial

--- GPU Status ---
GPU Detected: NVIDIA GeForce RTX 3090 Ti
VRAM: 23.68 GB

--- CPU Status ---
CPU Cores available: 24


In [34]:
import math
from pathlib import Path
from typing import List, Tuple

def split_lines_file(path: str | Path, train_ratio: float = 0.8, seed: int = 0):
    import random
    rng = random.Random(seed)
    lines = Path(path).read_text(encoding="utf-8", errors="ignore").splitlines()
    rng.shuffle(lines)
    n = len(lines)
    n_train = int(n * train_ratio)
    n_val = int(n * (1 - train_ratio) / 2)
    train, val, test = lines[:n_train], lines[n_train:n_train+n_val], lines[n_train+n_val:]
    def write(name, rows):
        p = Path(path).parent / f"{Path(path).stem}.{name}.txt"
        p.write_text("\n".join(rows), encoding="utf-8")
        return str(p)
    return write("train", train), write("val", val), write("test", test)

def compute_ppl_and_entropy(nll_sum: float, n_tokens: int) -> Tuple[float, float]:
    """NLL should be in NATS (natural log). Returns (Perplexity, Entropy_nats)."""
    if n_tokens == 0: return float("inf"), float("inf")
    avg_nll = nll_sum / n_tokens
    ppl = math.exp(avg_nll)
    entropy = avg_nll # Entropy H = Cross-Entropy when model matches empirical dist
    return ppl, entropy

def log10_to_nats(log10_val: float) -> float:
    return log10_val * math.log(10)

In [35]:
import textwrap

# A slightly richer corpus so statistical smoothing has enough data to work with
corpus_text = textwrap.dedent("""\
I love NLP .
I love speech recognition .
I love language models .
Language models are fun .
Speech recognition needs language models .
NLP stands for natural language processing .
Deep learning improved NLP significantly .
Transformers changed deep learning forever .
Recurrent neural networks were used for sequence modeling .
Long short term memory networks solved the vanishing gradient problem .
Attention is all you need .
Retrieval augmented generation helps reduce hallucinations .
Language models assign probabilities to sequences of words .
Perplexity is a common metric for evaluating language models .
Cross entropy measures the difference between two probability distributions .
Smoothing techniques help handle unseen words in n-gram models .
Backoff models fall back to lower order n-grams when higher orders are missing .
Kneser Ney smoothing is the state of the art for statistical language models .
Neural language models use continuous representations of words .
GPT models are autoregressive language models .
BERT is a masked language model .
Fine tuning adapts pretrained models to specific downstream tasks .
Large language models require massive amounts of training data .
Scaling laws dictate the relationship between model size and performance .
""".strip())

corpus_file = Path("lm_toy.txt")
corpus_file.write_text(corpus_text, encoding="utf-8")

# Split 80/10/10
train_f, val_f, test_f = split_lines_file(corpus_file)
print(f"Train lines: {len(Path(train_f).read_text().splitlines())}")
print(f"Test lines: {len(Path(test_f).read_text().splitlines())}")

Train lines: 19
Test lines: 3


In [36]:
!ls

bk	    general.log  lm_toy.test.txt   lm_toy.val.txt  transformer
data	    general.txt  lm_toy.train.txt  lstm
domain.log  kenlm	 lm_toy.txt	   tmp_kenlm


In [41]:
!wc *.txt

   2   24  151 lm_toy.test.txt
  18  151  977 lm_toy.train.txt
  23  198 1270 lm_toy.txt
   1   23  140 lm_toy.val.txt
  44  396 2538 total


In [37]:
import kenlm

arpa_file = "lm_toy_3gram.arpa"
tmp_dir = f"{WORK_DIR}/tmp_kenlm"
!mkdir -p {tmp_dir}

# FIX 1: Added '--discount_fallback' so it won't crash on small datasets
# FIX 2: Added '2>{arpa_file}.err' to catch any remaining errors in a file
!cat {train_f} | lmplz -o 3 -S 16G -T {tmp_dir} --discount_fallback > {arpa_file} 2>{arpa_file}.err

# Safety check: Make sure the file isn't empty before loading!
if Path(arpa_file).stat().st_size == 0:
    print("ERROR: ARPA file is empty! Check the error log:")
    print(Path(f"{arpa_file}.err").read_text())
else:
    model_kenlm = kenlm.LanguageModel(arpa_file)
    print(f"Successfully loaded {model_kenlm.order}-gram model.")

    def eval_kenlm(path: str):
        sum_log10, n_tokens = 0.0, 0
        for line in Path(path).read_text(encoding="utf-8").splitlines():
            if not line.strip(): continue
            for prob, length, oov in model_kenlm.full_scores(line):
                sum_log10 += prob
                n_tokens += 1
        sum_nats = log10_to_nats(sum_log10)
        ppl, entropy = compute_ppl_and_entropy(-sum_nats, n_tokens)
        return ppl, entropy

    ppl_kenlm, ent_kenlm = eval_kenlm(test_f)
    print(f"KenLM Test -> PPL: {ppl_kenlm:.2f}, Entropy: {ent_kenlm:.2f} nats")

Successfully loaded 3-gram model.
KenLM Test -> PPL: 25.80, Entropy: 3.25 nats


Loading the LM will be faster if you build a binary file.
Reading /mnt/disk1/ye/exp/LM-Tutorial/lm_toy_3gram.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


In [37]:
!ls

lm_toy_3gram.arpa      lm_toy.test.txt	 lm_toy.txt	 tmp_kenlm
lm_toy_3gram.arpa.err  lm_toy.train.txt  lm_toy.val.txt


## Learn ARPA Format

Kenlm နဲ့ ဆောက်ပြီးထွက်လာတဲ့ Language Model ဖိုင်ကို လေ့လာကြရအောင်။  



In [38]:
!cat ./lm_toy_3gram.arpa

\data\
ngram 1=102
ngram 2=140
ngram 3=143

\1-grams:
-2.14211	<unk>	0
0	<s>	-0.30103
-2.100283	</s>	0
-2.100283	Cross	-0.30103
-2.100283	entropy	-0.30103
-2.100283	measures	-0.30103
-1.6676633	the	-0.30103
-2.100283	difference	-0.30103
-1.7268959	between	-0.30103
-2.100283	two	-0.30103
-2.100283	probability	-0.30103
-2.100283	distributions	-0.30103
-1.0318743	.	-1
-2.100283	I	-0.30103
-2.100283	love	-0.30103
-2.14211	NLP	-0.30103
-2.100283	Kneser	-0.30103
-2.100283	Ney	-0.30103
-2.100283	smoothing	-0.30103
-2.14211	is	-0.30103
-2.100283	state	-0.30103
-2.14211	of	-0.30103
-2.100283	art	-0.30103
-1.7268959	for	-0.30103
-2.100283	statistical	-0.30103
-1.3673046	language	-0.50514996
-1.6676633	models	-0.30103
-2.100283	Transformers	-0.30103
-2.100283	changed	-0.30103
-2.100283	deep	-0.30103
-1.7268959	learning	-0.30103
-2.100283	forever	-0.30103
-2.100283	Deep	-0.30103
-2.100283	improved	-0.30103
-2.100283	significantly	-0.30103
-2.100283	Retrieval	-0.30103
-2.100283	augmented	-0.30103
-

In [42]:
!ls -lh

total 36K
-rw-rw-r-- 1 ye ye  12K May  1 12:50 lm_toy_3gram.arpa
-rw-rw-r-- 1 ye ye 1.2K May  1 12:50 lm_toy_3gram.arpa.err
-rw-rw-r-- 1 ye ye  151 May  1 12:48 lm_toy.test.txt
-rw-rw-r-- 1 ye ye  977 May  1 12:48 lm_toy.train.txt
-rw-rw-r-- 1 ye ye 1.3K May  1 12:48 lm_toy.txt
-rw-rw-r-- 1 ye ye  140 May  1 12:48 lm_toy.val.txt
drwxrwxr-x 2 ye ye 4.0K May  1 12:50 tmp_kenlm


**Error ဖိုင်ကိုလည်း လေ့လာကြည့်ရအောင်။**  

တကယ်က error ဖိုင်လို့ နာမည်ပေးခဲ့ပေမဲ့ build log ဖိုင်ပါ။ Python code ထဲမှာ  

`lmplz command > {arpa_file} 2>{arpa_file}.err` ဆိုပြီး သုံးခဲ့လို့ မော်ဒယ်ဖိုင်ကိုတော့ {arpa_file} အနေနဲ့ သိမ်းပေးပါ။ တခြား progress logs တွေ warning တွေနဲ့ error တွေကိုတော့ `{arpa_file}.err` မှာ သိမ်းပေးပါလို့ ရေးထားခဲ့လို့ ထွက်လာတဲ့ ဖိုင်ပါ။  

In [43]:
!cat lm_toy_3gram.arpa.err

=== 1/5 Counting and sorting n-grams ===
File stdin isn't normal.  Using slower read() instead of mmap().  No progress bar.
Unigram tokens 151 types 102
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:1224 2:5975606272 3:11204261888
Substituting fallback discounts for order 1: D1=0.5 D2=1 D3+=1.5
Substituting fallback discounts for order 2: D1=0.5 D2=1 D3+=1.5
Statistics:
1 102 D1=0.897959 D2=0.383673 D3+=3
2 140 D1=0.5 D2=1 D3+=1.5
3 143 D1=0.5 D2=1 D3+=1.5
Memory estimate for binary LM:
type       B
probing 8596 assuming -p 1.5
probing 9848 assuming -r models -p 1.5
trie    4563 without quantization
trie    6401 assuming -q 8 -b 8 quantization 
trie    4565 assuming -a 22 array pointer compression
trie    6403 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:1224 2:2240 3:2860
=== 4/5 Calculating and writing order-interpolated probabilities ===
Chain sizes: 1:1224 2:2240 3:2

ဒီနေရာမှာ ပြောနေတဲ့ 

```
Substituting fallback discounts for order 1: D1=0.5 D2=1 D3+=1.5
Substituting fallback discounts for order 2: D1=0.5 D2=1 D3+=1.5
```

message တွေက ဒေတာမှာ sparsity problem ရှိနေတယ် ဆိုတာကို ဆိုလိုတာပါ။ ဆရာတို့ ဒေတာဖိုင်က ဖိုင်အသေးလေးမို့လို့ပါ။ တကယ်တမ်း ngram LM ကို ဆောက်ဖို့က ngram pair တွေက ထပ်ခါထပ်ခါ ပါနေဖို့ လိုအပ်ပါတယ်။ အဲဒါမှာ LM ကောင်းကောင်း ရမှာပါ။  

မင်းရဲ့ data နဲ့ bigram, trigram counting တန်ဖိုးတွေက မငြိမ်ဘူး။ ပရမ်းပတာဖြစ်နေတယ်လို့ ဆိုလိုတာပါ။  
အဲဒါကြောင့် Kneser-Ney smoothing discounts ကို တွက်လို့ မရဘူး။ Default training အတိုင်းပဲ ဆက်လုပ်သွားမယ် စသည်ဖြင့် information ပေးတာပါ။  

## SRILM Installation

- Put the SRILM source under /home/ye/tool/srilm
- Compile it there (binaries will appear under $SRILM/bin/$MACHINE_TYPE)
- Add the binaries to your PATH by setting environment variables in your shell config.

This matches the official INSTALL instructions: unpack, set SRILM, run make World, then add `$SRILM/bin/$MACHINE_TYPE` and `$SRILM/bin to PATH`, and `$SRILM/man` to MANPATH.

### 0) Install OS prerequisites

SRILM needs: a C/C++ compiler (GCC), GNU make, gawk, gzip, and (importantly) a C‑shell (csh/tcsh) because the sbin/machine-type script used during the build is a csh script.  


In [ ]:
#sudo apt-get update
#sudo apt-get install -y build-essential gawk gzip bzip2 tcsh

Note: The FAQ explicitly says that without /bin/csh you’ll see “make: /sbin/machine-type: Command not found”; installing tcsh fixes this.

Verify key tools exist:

In [19]:
!which gcc g++ make gawk gzcat gunzip

/usr/lib/nvidia-cuda-toolkit/bin/gcc
/usr/lib/nvidia-cuda-toolkit/bin/g++
/usr/bin/make
/usr/bin/gunzip


In [22]:
!which csh   # should resolve to /bin/csh or a link to tcsh

အထက်မှာ မြင်ရတဲ့အတိုင်း ဆရာ့စက်ထဲမှာ csh ကို which command နဲ့ ခေါ်ကြည့်တော့ ဘာမှမပြလို့ အောက်ပါအတိုင်း terminal ထွက်ပြီး installation လုပ်ခဲ့တယ်။ Jupyter Notebook cell ထဲကနေက admin password ကို ရိုက်ပေးဖို့ခက်လို့...  

```bash
ye@lst-hpc3090:~/tool$ sudo apt-get install -y tcsh  
[sudo] password for ye:
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tcsh
0 upgraded, 1 newly installed, 0 to remove and 95 not upgraded.
Need to get 450 kB of archives.
After this operation, 1,418 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 tcsh amd64 6.24.10-4build1 [450 kB]
Fetched 450 kB in 2s (278 kB/s)
Selecting previously unselected package tcsh.
(Reading database ... 425519 files and directories currently installed.)
Preparing to unpack .../tcsh_6.24.10-4build1_amd64.deb ...
Unpacking tcsh (6.24.10-4build1) ...
Setting up tcsh (6.24.10-4build1) ...
update-alternatives: using /bin/tcsh to provide /bin/csh (csh) in auto mode
Processing triggers for debianutils (5.17build1) ...
Processing triggers for man-db (2.12.0-4build2) ...
ye@lst-hpc3090:~/tool$
ye@lst-hpc3090:~/tool$ which csh
/usr/bin/csh
ye@lst-hpc3090:~/tool$
```

SRILM Download Page:[http://www.speech.sri.com/projects/srilm/download.html](http://www.speech.sri.com/projects/srilm/download.html)  
SRILM GitHub Page: [https://github.com/BitSpeech/SRILM](https://github.com/BitSpeech/SRILM)

In [38]:
!tcsh --help

tcsh 6.24.10 (Astron) 2023-04-14 (x86_64-unknown-linux) options wide,nls,dl,al,kan,sm,rh,nd,color,filec

-b file		batch mode, read and execute commands from `file' 
-c command	run `command' from next argument 
-d		load directory stack from `~/.cshdirs' 
-Dname[=value]	define environment variable `name' to `value' (DomainOS only) 
-e		exit on any error 
-f		start faster by ignoring the start-up file 
-F		use fork() instead of vfork() when spawning (ConvexOS only) 
-i		interactive, even when input is not from a terminal 
-l		act as a login shell, must be the only option specified 
-m		load the start-up file, whether or not owned by effective user 
-n file		no execute mode, just check syntax of the following `file' 
-q		accept SIGQUIT for running under a debugger 
-s		read commands from standard input 
-t		read one line from standard input 
-v		echo commands after history substitution 
-V		like -v but including commands read from the start-up file 
-x		echo commands immediately before exe

In [39]:
%pwd

'/mnt/disk1/ye/exp/LM-Tutorial'

In [40]:
%cd /home/ye/tool/

/home/ye/tool


In [29]:
!git clone https://github.com/BitSpeech/SRILM

Cloning into 'SRILM'...
remote: Enumerating objects: 4543, done.
remote: Counting objects: 100% (336/336), done.
remote: Compressing objects: 100% (299/299), done.
remote: Total 4543 (delta 44), reused 99 (delta 31), pack-reused 4207 (from 1)
Receiving objects: 100% (4543/4543), 65.67 MiB | 18.78 MiB/s, done.
Resolving deltas: 100% (2304/2304), done.


the most common way to check the hardware architecture:  

In [31]:
!uname -m

x86_64


In [32]:
!lscpu | grep Architecture

Architecture:                            x86_64


**Updating the "SRILM path" inside "Makefile"**

```
# the following is the original example path of Makefile
# SRILM = /home/speech/stolcke/project/srilm/devel

# Add your path here, for example
SRILM = /home/ye/tool/SRILM
```

make command က Jupyter Notebook ရဲ့ cell အတွင်းက run ရင် အလုပ်မလုပ်တဲ့အခါရှိပါတယ်။ အကြောင်းအရင်းကတော့ bash environment variable, paths တွေနဲ့ shell behavior တွေက မတူကြလို့ပါ။   

ဒီ tutorial အတွက် SRILM ကို terminal မှာပဲ ထွက်ပြီး installation လုပ်ခဲ့တယ်။ အောက်ပါအတိုင်းပါ။  

```bash
ye@lst-hpc3090:~/tool/SRILM$ make SRILM=/home/ye/tool/SRILM World | tee srilm_install.log
...
...
...
make[2]: Entering directory '/home/ye/tool/SRILM/lm/src'
make[2]: Nothing to be done for 'release-scripts'.
make[2]: Leaving directory '/home/ye/tool/SRILM/lm/src'
make[2]: Entering directory '/home/ye/tool/SRILM/flm/src'
make[2]: Nothing to be done for 'release-scripts'.
make[2]: Leaving directory '/home/ye/tool/SRILM/flm/src'
make[2]: Entering directory '/home/ye/tool/SRILM/lattice/src'
make[2]: Nothing to be done for 'release-scripts'.
make[2]: Leaving directory '/home/ye/tool/SRILM/lattice/src'
make[2]: Entering directory '/home/ye/tool/SRILM/utils/src'
/home/ye/tool/SRILM/sbin/decipher-install 0555 change-lm-vocab ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 empty-sentence-lm ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 rescore-decipher ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 rescore-acoustic ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 rescore-reweight ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 rescore-minimize-wer ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 make-batch-counts ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 merge-batch-counts ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 make-big-lm ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 make-multiword-pfsg ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 pfsg-from-ngram ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 nbest-error ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 nbest-rover ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 search-rover-combo ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 rexport.gnumake ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 align-with-tags ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 compute-sclite ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 compute-sclite-nbest ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 compare-sclite ../../bin
/home/ye/tool/SRILM/sbin/decipher-install 0555 cumbin ../../bin
make[2]: Leaving directory '/home/ye/tool/SRILM/utils/src'
make[2]: Entering directory '/home/ye/tool/SRILM/zlib/src'
make[2]: Nothing to be done for 'release-scripts'.
make[2]: Leaving directory '/home/ye/tool/SRILM/zlib/src'
make[1]: Leaving directory '/home/ye/tool/SRILM'
ye@lst-hpc3090:~/tool/SRILM$
```

In [44]:
%pwd

'/home/ye/tool'

In [45]:
%cd SRILM

/home/ye/tool/SRILM


In [46]:
!ls

ACKNOWLEDGEMENTS  Copyright  include  License	misc	 srilm_install.log
bin		  doc	     INSTALL  lm	README	 utils
CHANGES		  dstruct    lattice  Makefile	RELEASE  visual_studio
common		  flm	     lib      man	sbin	 zlib


SRILM ကို make နဲ့ compile လုပ်တာက ကိုယ့်စက်ထဲမှာ အဆင်ပြေပြေနဲ့ ပြီးသွားရင် အောက်ပါလိုမျိုး statistical LM နဲ့ ပတ်သက်တဲ့ tool တွေကို တွေ့ရလိမ့်မယ်။  

In [49]:
!ls ./bin/i686-m64/* --color=auto -F

./bin/i686-m64/add-classes-to-pfsg*
./bin/i686-m64/add-dummy-bows*
./bin/i686-m64/add-pauses-to-pfsg*
./bin/i686-m64/add-ppls*
./bin/i686-m64/anti-ngram*
./bin/i686-m64/bytelog-to-log10*
./bin/i686-m64/classes-to-fsm*
./bin/i686-m64/combine-acoustic-scores*
./bin/i686-m64/combine-rover-controls*
./bin/i686-m64/compare-ppls*
./bin/i686-m64/compute-best-mix*
./bin/i686-m64/compute-best-rover-mix*
./bin/i686-m64/compute-best-sentence-mix*
./bin/i686-m64/compute-oov-rate*
./bin/i686-m64/concat-sausages*
./bin/i686-m64/context-ngrams*
./bin/i686-m64/continuous-ngram-count*
./bin/i686-m64/de-vq-lm*
./bin/i686-m64/disambig*
./bin/i686-m64/extract-skip-probs*
./bin/i686-m64/filter-event-counts*
./bin/i686-m64/find-reference-posteriors*
./bin/i686-m64/fix-ctm*
./bin/i686-m64/fngram*
./bin/i686-m64/fngram-count*
./bin/i686-m64/fsm-to-pfsg*
./bin/i686-m64/get-gt-counts*
./bin/i686-m64/get-unigram-probs*
./bin/i686-m64/hidden-ngram*
./bin/i686-m64/hits-from-log*
./bin/i686-m64/htklat-vocab*
./bin/

**updating your .bashrc file**  

```
export PATH="/home/ye/tool/SRILM/bin/i686-m64:$PATH"
```

ပြီးရင် .bashrc ကို ပြန် activate လုပ်ပါ။  

```bash
source /home/ye/.bashrc
```

In [1]:
%cd /mnt/disk1/ye/exp/LM-Tutorial

/mnt/disk1/ye/exp/LM-Tutorial


In [2]:
!ngram-count --help

/bin/bash: line 1: ngram-count: command not found


In [3]:
!echo $PATH

/home/ye/tool/speech_tools/bin:/home/ye/tool/festival/bin:/home/ye/tool/festvox/bin:/home/ye/tool/flite/bin:/home/ye/.nvm/versions/node/v20.19.3/bin:/usr/lib/nvidia-cuda-toolkit/bin:/usr/bin:/bin:/home/ye/tool/marian/build:/home/ye/.local/bin:/home/ye/tool/speech_tools/bin:/home/ye/tool/festival/bin:/home/ye/tool/festvox/bin:/home/ye/tool/flite/bin:/home/ye/.nvm/versions/node/v20.19.3/bin:/home/ye/.cargo/bin:/usr/lib/nvidia-cuda-toolkit/bin:/usr/bin:/bin:/home/ye/tool/marian/build:/home/ye/miniforge3/condabin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin:/home/ye/.local/bin:/home/ye/tool/kenlm/build/bin/:/home/ye/.local/bin:/home/ye/.local/bin:/home/ye/tool/kenlm/build/bin/


အောက်ပါ command က တမင်တကာ run ပြတာ။ source ကို activate လုပ်လည်း updated PATH က ရမလာဘူး။  

In [4]:
!source /home/ye/.bashrc

In [5]:
!echo $PATH

/home/ye/tool/speech_tools/bin:/home/ye/tool/festival/bin:/home/ye/tool/festvox/bin:/home/ye/tool/flite/bin:/home/ye/.nvm/versions/node/v20.19.3/bin:/usr/lib/nvidia-cuda-toolkit/bin:/usr/bin:/bin:/home/ye/tool/marian/build:/home/ye/.local/bin:/home/ye/tool/speech_tools/bin:/home/ye/tool/festival/bin:/home/ye/tool/festvox/bin:/home/ye/tool/flite/bin:/home/ye/.nvm/versions/node/v20.19.3/bin:/home/ye/.cargo/bin:/usr/lib/nvidia-cuda-toolkit/bin:/usr/bin:/bin:/home/ye/tool/marian/build:/home/ye/miniforge3/condabin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin:/home/ye/.local/bin:/home/ye/tool/kenlm/build/bin/:/home/ye/.local/bin:/home/ye/.local/bin:/home/ye/tool/kenlm/build/bin/


In [44]:
import os
# This updates the PATH for the duration of the notebook session
os.environ['PATH'] = "/home/ye/tool/SRILM/bin/i686-m64:" + os.environ['PATH']

# Verify
!echo $PATH

/home/ye/tool/SRILM/bin/i686-m64:/home/ye/tool/speech_tools/bin:/home/ye/tool/festival/bin:/home/ye/tool/festvox/bin:/home/ye/tool/flite/bin:/home/ye/.nvm/versions/node/v20.19.3/bin:/usr/lib/nvidia-cuda-toolkit/bin:/usr/bin:/bin:/home/ye/tool/marian/build:/home/ye/.local/bin:/home/ye/tool/speech_tools/bin:/home/ye/tool/festival/bin:/home/ye/tool/festvox/bin:/home/ye/tool/flite/bin:/home/ye/.nvm/versions/node/v20.19.3/bin:/home/ye/.cargo/bin:/usr/lib/nvidia-cuda-toolkit/bin:/usr/bin:/bin:/home/ye/tool/marian/build:/home/ye/miniforge3/condabin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin:/home/ye/.local/bin:/home/ye/tool/kenlm/build/bin/:/home/ye/.local/bin:/home/ye/.local/bin:/home/ye/tool/kenlm/build/bin/


အထက်မှာ မြင်ရတဲ့အတိုင်း `/home/ye/tool/SRILM/bin/i686-m64` ကို PATH environment variable မှာ သိသွားပြီမို့လို့ SRILM command တွေကို ခုဆိုရင်တော့ ခေါ် run လို့ ရလိမ့်မယ်။  

In [45]:
!ngram-count -help

Usage of command "ngram-count"
 -version:                 print version information
 -order:                   max ngram order
		Default value: 3
 -varprune:                pruning threshold for variable order ngrams
		Default value: 0
 -debug:                   debugging level for LM
		Default value: 0
 -recompute:               recompute lower-order counts by summation
 -sort:                    sort ngrams output
 -write-order:             output ngram counts order
		Default value: 0
 -tag:                     file tag to use in messages
 -text:                    text file to read
 -text-has-weights:        text file contains count weights
 -text-has-weights-last:   text file contains count weights at ends of lines
 -no-sos:                  don't insert start-of-sentence tokens
 -no-eos:                  don't insert end-of-sentence tokens
 -read:                    counts file to read
 -intersect:               intersect counts with this file
 -read-with-mincounts:     apply mini

In [47]:
%cd /mnt/disk1/ye/exp/LM-Tutorial

/mnt/disk1/ye/exp/LM-Tutorial


In [45]:
!ls

lm_toy_3gram.arpa      lm_toy.test.txt	 lm_toy.txt	 tmp_kenlm
lm_toy_3gram.arpa.err  lm_toy.train.txt  lm_toy.val.txt


In [48]:
# Train KN-smoothed 3-gram
!ngram-count -order 3 -kndiscount -text lm_toy.train.txt  -lm lm_toy_srilm.arpa

one of required modified KneserNey count-of-counts is zero
error in discount estimator for order 1


In [48]:
!ls

lm_toy_3gram.arpa      lm_toy.test.txt	 lm_toy.txt	 tmp_kenlm
lm_toy_3gram.arpa.err  lm_toy.train.txt  lm_toy.val.txt


## Why did SRILM crash?

Modified Kneser-Ney (KN) smoothing calculates "count-of-counts" (how many words appear exactly once, exactly twice, exactly three times, etc.) to figure out how much probability mass to steal from seen words to give to unseen words.  

Because your training data is so small, one of those counts (likely the number of bigrams/trigrams that appear exactly twice) is exactly zero. The KN math formula involves dividing by that number. Dividing by zero causes the C++ code to crash hard.  

Compare this to KenLM: Remember how KenLM printed Substituting fallback discounts... in your .err file and kept running? KenLM has modern "training wheels" for tiny datasets. SRILM is older, stricter research code—it sees a divide-by-zero coming and immediately halts to protect the mathematical integrity of the model.  

## Using Witten-Bell Discounting

When data is too small for Kneser-Ney, the standard fallback recommended by the SRILM creators is Witten-Bell smoothing (-wbdiscount). Instead of relying on complex count-of-counts, Witten-Bell estimates the probability of seeing a new word based on how many unique words you've seen so far. It never divides by zero. 

In [49]:
!ngram-count -order 3 -wbdiscount -text lm_toy.train.txt  -lm lm_toy_srilm.arpa

In [50]:
!ls

lm_toy_3gram.arpa      lm_toy_srilm.arpa  lm_toy.train.txt  lm_toy.val.txt
lm_toy_3gram.arpa.err  lm_toy.test.txt	  lm_toy.txt	    tmp_kenlm


SRILM နဲ့ ဆောက်ထားတဲ့ language model တော့ ရလာပြီ။  
Evaluation လုပ်ကြည့်ရအောင်။  

In [50]:
!head -n 30 ./lm_toy_srilm.arpa


\data\
ngram 1=101
ngram 2=140
ngram 3=3

\1-grams:
-1.130334	.	-1.267606
-1.130334	</s>
-99	<s>	-0.2451774
-2.130334	Attention	-0.2945479
-2.130334	BERT	-0.2945479
-2.130334	Cross	-0.297801
-2.130334	Deep	-0.2961775
-2.130334	Fine	-0.297801
-2.130334	GPT	-0.282966
-1.954242	I	-0.4722688
-2.130334	Kneser	-0.297801
-2.130334	Language	-0.282966
-2.130334	Large	-0.2863067
-2.130334	Long	-0.297801
-1.829304	NLP	-0.2606013
-2.130334	Neural	-0.2863067
-2.130334	Ney	-0.297801
-2.130334	Retrieval	-0.297801
-2.130334	Scaling	-0.297801
-2.130334	Smoothing	-0.297801
-2.130334	Speech	-0.297801
-2.130334	Transformers	-0.297801
-2.130334	a	-0.297801


In [51]:
!cat lm_toy.test.txt

I love speech recognition .
Perplexity is a common metric for evaluating language models .
Language models assign probabilities to sequences of words .

In [52]:
# Evaluate
!ngram -lm lm_toy_srilm.arpa -ppl lm_toy.test.txt

file lm_toy.test.txt: 3 sentences, 24 words, 8 OOVs
0 zeroprobs, logprob= -18.98879 ppl= 9.986423 ppl1= 15.37444


## Reading the ARPA Format  

Notice the structure of this file. This is the standard ARPA format (used by SRILM, KenLM, and IRSTLM):  

1. **\\data\\:** Metadata telling us how many N-grams exist in the file.  
2. **\\1-grams:** Unigram probabilities.  
    - Format: log10(Prob) Word [log10(Backoff_Weight)]  
    - Example: -1.39794 \<unk\> -0.477121 means P() = 10^-1.39.
    - The backoff weight is used if the unigram doesn't appear in a bigram context.  
3. **\\2-grams and \\3-grams**: Higher order probabilities.
4. **\\end\\:** The end of the file.

**Key insight:** Everything is in log base 10 (to prevent underflow with tiny probabilities).

- A log prob of -1.0 means a 10% probability.
- A log prob of -2.0 means a 1% probability.

## Tracing with -debug 2 Option

How did SRILM get PPL = 9.986? Let's force it to show its work.  

In [54]:
# -debug 2 prints the exact log-probability calculated for EVERY SINGLE WORD
!ngram -lm lm_toy_srilm.arpa -ppl lm_toy.test.txt -debug 2

reading 101 1-grams
reading 140 2-grams
reading 3 3-grams
I love speech recognition .
	p( I | <s> ) 	= [2gram] 0.05405402 [ -1.267172 ]
	p( love | I ...) 	= [3gram] 0.6666666 [ -0.1760913 ]
	p( <unk> | love ...) 	= [OOV] 0 [ -inf ]
	p( recognition | <unk> ...) 	= [1gram] 0.007407405 [ -2.130334 ]
	p( . | recognition ...) 	= [1gram] 0.03731341 [ -1.428135 ]
	p( </s> | . ...) 	= [2gram] 0.95 [ -0.02227639 ]
1 sentences, 5 words, 1 OOVs
0 zeroprobs, logprob= -5.024009 ppl= 10.11118 ppl1= 18.03027

Perplexity is a common metric for evaluating language models .
	p( <unk> | <s> ) 	= [OOV] 0 [ -inf ]
	p( is | <unk> ...) 	= [1gram] 0.01481481 [ -1.829304 ]
	p( a | is ...) 	= [2gram] 0.1666667 [ -0.7781513 ]
	p( <unk> | a ...) 	= [OOV] 0 [ -inf ]
	p( <unk> | <unk> ...) 	= [OOV] 0 [ -inf ]
	p( for | <unk> ...) 	= [1gram] 0.01111112 [ -1.954242 ]
	p( <unk> | for ...) 	= [OOV] 0 [ -inf ]
	p( language | <unk> ...) 	= [1gram] 0.03333335 [ -1.477121 ]
	p( models | language ...) 	= [2gram] 0.5454546 [

In [54]:
!ngram -lm lm_toy_srilm.arpa -ppl lm_toy.test.txt -debug 1

reading 101 1-grams
reading 140 2-grams
reading 3 3-grams
I love speech recognition .
1 sentences, 5 words, 1 OOVs
0 zeroprobs, logprob= -5.024009 ppl= 10.11118 ppl1= 18.03027

Perplexity is a common metric for evaluating language models .
1 sentences, 10 words, 4 OOVs
0 zeroprobs, logprob= -6.733423 ppl= 9.160467 ppl1= 13.25065

Language models assign probabilities to sequences of words .
1 sentences, 9 words, 3 OOVs
0 zeroprobs, logprob= -7.231358 ppl= 10.79074 ppl1= 16.04081

file lm_toy.test.txt: 3 sentences, 24 words, 8 OOVs
0 zeroprobs, logprob= -18.98879 ppl= 9.986423 ppl1= 15.37444


In [55]:
!ngram -lm lm_toy_srilm.arpa -ppl lm_toy.test.txt -debug 3

reading 101 1-grams
reading 140 2-grams
reading 3 3-grams
I love speech recognition .
	p( I | <s> ) 	= [2gram] 0.05405402 [ -1.267172 ] / 0.9999995
	p( love | I ...) 	= [3gram] 0.6666666 [ -0.1760913 ] / 0.9999998
	p( <unk> | love ...) 	= [OOV] 0 [ -inf ] / 1
	p( recognition | <unk> ...) 	= [1gram] 0.007407405 [ -2.130334 ] / 0.9999998
	p( . | recognition ...) 	= [1gram] 0.03731341 [ -1.428135 ] / 0.9999999
	p( </s> | . ...) 	= [2gram] 0.95 [ -0.02227639 ] / 1
1 sentences, 5 words, 1 OOVs
0 zeroprobs, logprob= -5.024009 ppl= 10.11118 ppl1= 18.03027

Perplexity is a common metric for evaluating language models .
	p( <unk> | <s> ) 	= [OOV] 0 [ -inf ] / 0.9999995
	p( is | <unk> ...) 	= [1gram] 0.01481481 [ -1.829304 ] / 0.9999998
	p( a | is ...) 	= [2gram] 0.1666667 [ -0.7781513 ] / 0.9999999
	p( <unk> | a ...) 	= [OOV] 0 [ -inf ] / 0.9999999
	p( <unk> | <unk> ...) 	= [OOV] 0 [ -inf ] / 0.9999998
	p( for | <unk> ...) 	= [1gram] 0.01111112 [ -1.954242 ] / 0.9999998
	p( <unk> | for ...) 	= 

## Debugging the Perplexity Calculation

Look closely at the debug output. For every single word, SRILM shows its work. Notice the tags like [3gram], [2gram], and [1gram]. This is Backoff in action!  

Example 1: "I love speech recognition ."  

p( I | \<s\> ) = [2gram]: It found the bigram \<s\> I in the training data.  
p( love | I ... ) = [3gram]: It found the trigram I love ... (likely I love NLP or similar in training) and used it to predict "love".  
p( \<unk\> | love ... ) = [OOV] 0 [ -inf ]: The word "speech" was not in our training vocabulary. SRILM replaces it with \<unk\>.  

Example 2: The OOV ChainLook at the second sentence: Perplexity is a common metric...  

Perplexity becomes \<unk\>. Log prob: [ -inf ]  
common becomes \<unk\>. Log prob: [ -inf ]  
metric becomes \<unk\>. Log prob: [ -inf ]  
Notice what happens to the word is: p( is | \<unk\> ... ) = [1gram] 0.01481481 [ -1.829304 ] Because \<unk\> is never appeared in the training data, the Bigram failed. The Trigram failed. It backed off all the way to the Unigram probability of "is"!  

## The "Magic" of SRILM's PPL Math

You see a lot of [ -inf ] in that log. If you add -infinity to anything, the total becomes -infinity, and Perplexity becomes Infinity. But look at the final math at the bottom:logprob= -18.98879 ppl= 9.986423  

How did SRILM avoid Infinity? Look at the denominator it used: 3 sentences, 24 words, 8 OOVs.SRILM calculates PPL as: 10^( -logprob / (words - OOVs + sentences) )  

Numerator: -18.98879  
Denominator: 24 - 8 + 3 = 19 (It completely ignores the 8 OOV words!)  

By excluding OOVs from the denominator and skipping their -inf scores from the numerator, SRILM gives the LM the "benefit of the doubt." It only judges the model on the words it actually had a chance to learn.  

## Extracting Raw N-gram Counts

In [56]:
# We can tell ngram-count to JUST count, and NOT estimate a language model
!ngram-count -order 3 -text lm_toy.train.txt -write lm_toy.raw_counts -write-vocab lm_toy.vocab

In [58]:
!ls

lm_toy_3gram.arpa      lm_toy_srilm.arpa  lm_toy.txt	  tmp_kenlm
lm_toy_3gram.arpa.err  lm_toy.test.txt	  lm_toy.val.txt
lm_toy.raw_counts      lm_toy.train.txt   lm_toy.vocab


In [57]:
!wc lm_toy.raw_counts

 384 1194 6200 lm_toy.raw_counts


In [70]:
!wc lm_toy.vocab

103 103 757 lm_toy.vocab


In [58]:
!head -20 lm_toy.raw_counts

stands	1
stands for	1
stands for natural	1
<s>	19
<s> Neural	1
<s> Neural language	1
<s> Cross	1
<s> Cross entropy	1
<s> Large	1
<s> Large language	1
<s> Long	1
<s> Long short	1
<s> Fine	1
<s> Fine tuning	1
<s> I	2
<s> I love	2
<s> NLP	1
<s> NLP stands	1
<s> Kneser	1
<s> Kneser Ney	1


In [59]:
# Look at the vocabulary size
!echo "--- Vocabulary ---"
!cat lm_toy.vocab

--- Vocabulary ---
-pau-
.
</s>
<s>
<unk>
Attention
BERT
Cross
Deep
Fine
GPT
I
Kneser
Language
Large
Long
NLP
Neural
Ney
Retrieval
Scaling
Smoothing
Speech
Transformers
a
adapts
all
amounts
and
are
art
augmented
autoregressive
between
changed
continuous
data
deep
dictate
difference
distributions
downstream
entropy
for
forever
fun
generation
gradient
hallucinations
handle
help
helps
improved
in
is
language
laws
learning
love
masked
massive
measures
memory
model
models
n-gram
natural
need
needs
networks
of
performance
pretrained
probability
problem
processing
recognition
reduce
relationship
representations
require
short
significantly
size
smoothing
solved
specific
stands
state
statistical
tasks
techniques
term
the
to
training
tuning
two
unseen
use
vanishing
words
you


In [60]:
# Look at the top 20 most frequent N-grams overall
# (In SRILM raw counts, unigrams have NO spaces, bigrams have 1 space, trigrams have 2 spaces)
!echo "--- Top 20 N-grams (Sorted by Count) ---"
!sort -nr -k2 lm_toy.raw_counts | head -n 20

--- Top 20 N-grams (Sorted by Count) ---
<s>	19
</s>	19
.	19
models	10
language	8
the	5
of	3
NLP	3
is	3
words	2
model	2
love	2
learning	2
I	2
for	2
between	2
are	2
you	1
vanishing	1
use	1


## Raw Counts vs. Smoothed Probabilities (The Root of the Problem)

Let's analyze the raw counts file.  

We have 384 lines of N-grams extracted from just ~20 sentences.  
Our vocabulary size is 102 words.  
Look at the Top 20 N-grams from our sorted output. What is the highest count? \<s\> (sentence start) has a count of 19. The highest bigram is \<s\> I with a count of 2.  

Almost every other bigram and trigram in this entire file has a count of exactly 1.  

The Maximum Likelihood Estimation (MLE) Trap  
If we didn't use smoothing (like Witten-Bell or Kneser-Ney), we would calculate probabilities using pure   MLE: `P(word | context) = Count(context, word) / Count(context)`  

If the test set asks for the probability of the bigram language models, MLE finds a count of 1. But what if the test set asks for the bigram models assign? It's not in this file (count is 0). MLE gives it a probability of exactly 0.0.  

If even one word in a sentence gets a probability of 0, the entire sentence probability becomes 0. The log probability becomes -infinity. The Perplexity becomes Infinity. The language model is broken.  

Do you remember why the word is had to back off to a [1gram] in our debug output? It was because the bigram \<unk\> is had a count of 0 in this exact raw counts file!


## Vocabulary Management (change-lm-vocab)

This is the most important practical tool. In real ASR/MT, you cannot have OOVs because your decoder dictionary is fixed. You must "force" the LM to match your dictionary.  

In [61]:
%%bash
# Let's artificially create a restricted vocabulary (e.g., an ASR pronunciation dictionary)
# We will remove some words that might appear in the test set
echo "Creating a restricted vocabulary..."
cat > lm_toy.restricted.vocab << EOF
<unk>
<s>
</s>
.
language
models
are
fun
I
love
NLP
EOF

Creating a restricted vocabulary...


In [62]:
!ls lm_toy.restricted.vocab

lm_toy.restricted.vocab


In [63]:
# change-lm-vocab re-maps the LM to ONLY use words in this list
# Any word not in the list becomes <unk>
!/home/ye/tool/SRILM/bin/change-lm-vocab -vocab lm_toy.restricted.vocab -lm lm_toy_srilm.arpa -write-lm lm_toy.restricted.arpa

!echo "\n--- Evaluating the Restricted LM ---"
# Notice the OOV count now!
!ngram -lm lm_toy.restricted.arpa -ppl lm_toy.test.txt

-: line 38: warning: 10 1-grams read, expected 101
-: line 38: warning: 12 2-grams read, expected 140
\n--- Evaluating the Restricted LM ---
file lm_toy.test.txt: 3 sentences, 24 words, 16 OOVs
0 zeroprobs, logprob= -5.667527 ppl= 3.275139 ppl1= 5.110236


## gawk Installation

အထက်ပါအတိုင်း gawk က စက်ထဲမှာ မရှိလို့ terminal ထွက်ပြီး installation လုပ်ခဲ့တယ်။  

```bash
ye@lst-hpc3090:/mnt/disk1/ye/exp/LM-Tutorial$ sudo apt-get install gawk
[sudo] password for ye:
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libsigsegv2
Suggested packages:
  gawk-doc
The following NEW packages will be installed:
  gawk libsigsegv2
0 upgraded, 2 newly installed, 0 to remove and 95 not upgraded.
Need to get 478 kB of archives.
After this operation, 1,826 kB of additional disk space will be used.
Do you want to continue? [Y/n] Y
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 libsigsegv2 amd64 2.14-1ubuntu2 [15.0 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 gawk amd64 1:5.2.1-2build3 [463 kB]
Fetched 478 kB in 2s (256 kB/s)
Selecting previously unselected package libsigsegv2:amd64.
(Reading database ... 425560 files and directories currently installed.)
Preparing to unpack .../libsigsegv2_2.14-1ubuntu2_amd64.deb ...
Unpacking libsigsegv2:amd64 (2.14-1ubuntu2) ...
Setting up libsigsegv2:amd64 (2.14-1ubuntu2) ...
Selecting previously unselected package gawk.
(Reading database ... 425567 files and directories currently installed.)
Preparing to unpack .../gawk_1%3a5.2.1-2build3_amd64.deb ...
Unpacking gawk (1:5.2.1-2build3) ...
Setting up gawk (1:5.2.1-2build3) ...
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.7) ...
ye@lst-hpc3090:/mnt/disk1/ye/exp/LM-Tutorial$
```

In [75]:
# change-lm-vocab re-maps the LM to ONLY use words in this list
# Any word not in the list becomes <unk>
!/home/ye/tool/SRILM/bin/change-lm-vocab -vocab lm_toy.restricted.vocab -lm lm_toy_srilm.arpa -write-lm lm_toy.restricted.arpa

!echo "\n--- Evaluating the Restricted LM ---"
# Notice the OOV count now!
!ngram -lm lm_toy.restricted.arpa -ppl lm_toy.test.txt

-: line 38: warning: 10 1-grams read, expected 101
-: line 38: warning: 12 2-grams read, expected 140
\n--- Evaluating the Restricted LM ---
file lm_toy.test.txt: 3 sentences, 24 words, 16 OOVs
0 zeroprobs, logprob= -5.667527 ppl= 3.275139 ppl1= 5.110236


## Solving the OOV Problem (And Exposing a Metric Loophole)

Compare the output of this evaluation to your original one.

Original: 8 OOVs, logprob= -18.98, ppl= 9.986  
Restricted: 16 OOVs, logprob= -5.66, ppl= 3.275    

Wait, the PPL went down significantly (9.98 -> 3.27). Does that mean our restricted model is actually better at predicting text?  

Absolutely not. This is a mathematical illusion. Let's look at the math to see why SRILM is lying to us.


## The Loophole in SRILM's Math

Remember the SRILM PPL formula: PPL = 10^( -logprob / (words - OOVs + sentences) )  

Let's calculate the denominators for both:  

Original Denominator: 24 words - 8 OOVs + 3 sentences = 19  
Restricted Denominator: 24 words - 16 OOVs + 3 sentences = 11  
Now look at the logprob (the numerator). SRILM completely ignores OOV words when calculating logprob. They contribute 0 to the sum.  

In the restricted test set, 16 out of 24 words are OOVs!  
That means SRILM is only doing math for 8 words total (like "I", "love", "language", "models").  
Because those 8 remaining words are very common, their probabilities are very high, resulting in a very small logprob (-5.66).  

The Illusion: By stripping out vocabulary, we drastically shrank both the numerator and the denominator. But because the numerator dropped faster than the denominator, the final PPL fraction artificially decreased.  

Analogy: Imagine a teacher gives a 20-question test, but a student only answers 5 questions (all correctly) and leaves 15 blank. If the teacher's grading formula ignores the blank questions entirely, the student looks like a genius with a perfect score, but in reality, they failed the test because they couldn't answer 75% of the material.


## Why do we still do this in the real world?

If change-lm-vocab breaks the PPL metric, why is it the most important tool in Speech Recognition (Kaldi, Whisper)?  

Because in ASR, vocabulary alignment is mandatory. Your acoustic model can only output sounds that exist in your pronunciation dictionary. If "Transformers" isn't in the dictionary, the acoustic model will physically never output that word.  

If your Language Model contains "Transformers" but your acoustic model cannot output it, their scores are misaligned during decoding, and the system crashes or produces garbage. You must use change-lm-vocab to strip the LM down to match the dictionary exactly.

We accept the "broken" PPL metric because the alternative (decoder crashes) is worse. In real ASR pipelines, we evaluate the final Word Error Rate (WER) of the audio transcription, not the isolated PPL of the LM!  

## The "Big LM" Pipeline (Conceptual)  

**"What if I have 100GB of text?"** You cannot run standard ngram-count on 100GB of text, it will run out of RAM. This cell introduces the "MapReduce-style" tools.  

```bash
# STEP 1: Split the counting into smaller chunks (batches)
# This uses disk instead of RAM to avoid Out-Of-Memory crashes
make-batch-counts -n 3 -order 5 -text massive_corpus.txt -dir batch_counts/

# STEP 2: Merge all the batch counts back together
merge-batch-counts -dir batch_counts/ -write massive_counts.gz

# STEP 3: Estimate the LM from the merged counts
# (make-big-lm is a wrapper that handles KN smoothing math for huge files)
make-big-lm -read massive_counts.gz -lm massive_5gram.arpa
```

## Scaling to Web-Scale Data

Look at the tools in your SRILM folder (e.g. /home/ye/tool/SRILM/bin/): make-batch-counts, merge-batch-counts, make-big-lm.  

Standard ngram-count loads all N-grams into RAM. A 5-gram model on 10B words requires ~128GB of RAM. SRILM solved this in the early 2000s using a "MapReduce-like" approach before Hadoop existed:  

1. make-batch-counts: Splits the text into chunks, counts N-grams in RAM, writes to disk.  
2. merge-batch-counts: Does a disk-based merge sort of all the chunk counts.  
3. make-big-lm: Reads the sorted counts in a single stream and calculates the complex Kneser-Ney discounts.  

This is why SRILM was the king of N-grams for 20 years. KenLM later made this even faster with better C++ multi-threading, but the architecture is the same.

## LM Interpolation (compute-best-mix)  

In the real world, you never use just ONE LM. You mix a Domain LM (e.g., Medical terms) with a General LM (e.g., General English).  

```bash
# To do this properly, we need two LMs. 
# Let's fake a "General" LM by training on just one sentence.
echo "This is a general sentence about the world ." > general.txt
!ngram-count -wbdiscount -order 3 -text general.txt -lm general.arpa

# Now we find the optimal mathematical weight (lambda) to mix them
# compute-best-mix reads the debug logs of both LMs on a tuning set to find the exact weight
!ngram -lm lm_toy_srilm.arpa -ppl lm_toy.test.txt -debug 2 > domain.log
!ngram -lm general.arpa -ppl lm_toy.test.txt -debug 2 > general.log

echo "--- Computing optimal interpolation weights ---"
!compute-best-mix domain.log general.log
```

လက်တွေ့ run ကြည့်ရအောင်။  

In [65]:
!echo "This is a general sentence about the world ." > general.txt
!ngram-count -wbdiscount -order 3 -text general.txt -lm general.arpa

In [66]:
!cat general.arpa


\data\
ngram 1=11
ngram 2=10
ngram 3=0

\1-grams:
-1	.	-0.2552725
-1	</s>
-99	<s>	-0.2552725
-1	This	-0.2552725
-1	a	-0.2552725
-1	about	-0.2552725
-1	general	-0.2552725
-1	is	-0.2552725
-1	sentence	-0.2552725
-1	the	-0.2552725
-1	world	-0.2552725

\2-grams:
-0.30103	. </s>
-0.30103	<s> This
-0.30103	This is
-0.30103	a general
-0.30103	about the
-0.30103	general sentence
-0.30103	is a
-0.30103	sentence about
-0.30103	the world
-0.30103	world .

\3-grams:

\end\


In [67]:
# Now we find the optimal mathematical weight (lambda) to mix them
# compute-best-mix reads the debug logs of both LMs on a tuning set to find the exact weight
!ngram -lm lm_toy_srilm.arpa -ppl lm_toy.test.txt -debug 2 > domain.log
!ngram -lm general.arpa -ppl lm_toy.test.txt -debug 2 > general.log

reading 101 1-grams
reading 140 2-grams
reading 3 3-grams
reading 11 1-grams
reading 10 2-grams
reading 0 3-grams


In [69]:
!cat domain.log

I love speech recognition .
	p( I | <s> ) 	= [2gram] 0.05405402 [ -1.267172 ]
	p( love | I ...) 	= [3gram] 0.6666666 [ -0.1760913 ]
	p( <unk> | love ...) 	= [OOV] 0 [ -inf ]
	p( recognition | <unk> ...) 	= [1gram] 0.007407405 [ -2.130334 ]
	p( . | recognition ...) 	= [1gram] 0.03731341 [ -1.428135 ]
	p( </s> | . ...) 	= [2gram] 0.95 [ -0.02227639 ]
1 sentences, 5 words, 1 OOVs
0 zeroprobs, logprob= -5.024009 ppl= 10.11118 ppl1= 18.03027

Perplexity is a common metric for evaluating language models .
	p( <unk> | <s> ) 	= [OOV] 0 [ -inf ]
	p( is | <unk> ...) 	= [1gram] 0.01481481 [ -1.829304 ]
	p( a | is ...) 	= [2gram] 0.1666667 [ -0.7781513 ]
	p( <unk> | a ...) 	= [OOV] 0 [ -inf ]
	p( <unk> | <unk> ...) 	= [OOV] 0 [ -inf ]
	p( for | <unk> ...) 	= [1gram] 0.01111112 [ -1.954242 ]
	p( <unk> | for ...) 	= [OOV] 0 [ -inf ]
	p( language | <unk> ...) 	= [1gram] 0.03333335 [ -1.477121 ]
	p( models | language ...) 	= [2gram] 0.5454546 [ -0.2632414 ]
	p( . | models ...) 	= [3gram] 0.4444445 [ -

In [70]:
!cat general.log

I love speech recognition .
	p( <unk> | <s> ) 	= [OOV] 0 [ -inf ]
	p( <unk> | <unk> ...) 	= [OOV] 0 [ -inf ]
	p( <unk> | <unk> ...) 	= [OOV] 0 [ -inf ]
	p( <unk> | <unk> ...) 	= [OOV] 0 [ -inf ]
	p( . | <unk> ...) 	= [1gram] 0.1 [ -1 ]
	p( </s> | . ...) 	= [2gram] 0.5 [ -0.30103 ]
1 sentences, 5 words, 4 OOVs
0 zeroprobs, logprob= -1.30103 ppl= 4.472136 ppl1= 20

Perplexity is a common metric for evaluating language models .
	p( <unk> | <s> ) 	= [OOV] 0 [ -inf ]
	p( is | <unk> ...) 	= [1gram] 0.1 [ -1 ]
	p( a | is ...) 	= [2gram] 0.5 [ -0.30103 ]
	p( <unk> | a ...) 	= [OOV] 0 [ -inf ]
	p( <unk> | <unk> ...) 	= [OOV] 0 [ -inf ]
	p( <unk> | <unk> ...) 	= [OOV] 0 [ -inf ]
	p( <unk> | <unk> ...) 	= [OOV] 0 [ -inf ]
	p( <unk> | <unk> ...) 	= [OOV] 0 [ -inf ]
	p( <unk> | <unk> ...) 	= [OOV] 0 [ -inf ]
	p( . | <unk> ...) 	= [1gram] 0.1 [ -1 ]
	p( </s> | . ...) 	= [2gram] 0.5 [ -0.30103 ]
1 sentences, 10 words, 7 OOVs
0 zeroprobs, logprob= -2.60206 ppl= 4.472136 ppl1= 7.368063

Language models

In [71]:
!echo "--- Computing optimal interpolation weights ---"
!compute-best-mix domain.log general.log

--- Computing optimal interpolation weights ---
iteration 1, lambda = (0.5 0.5), ppl = 14.1176
iteration 2, lambda = (0.795617 0.204383), ppl = 10.9167
iteration 3, lambda = (0.894906 0.105094), ppl = 10.356
iteration 4, lambda = (0.937108 0.0628916), ppl = 10.1782
iteration 5, lambda = (0.958959 0.0410407), ppl = 10.1009
iteration 6, lambda = (0.971775 0.0282248), ppl = 10.0607
iteration 7, lambda = (0.979921 0.020079), ppl = 10.0372
iteration 8, lambda = (0.985386 0.0146145), ppl = 10.0223
iteration 9, lambda = (0.989191 0.0108085), ppl = 10.0124
iteration 10, lambda = (0.991914 0.00808586), ppl = 10.0056
iteration 11, lambda = (0.9939 0.00609991), ppl = 10.0007
iteration 12, lambda = (0.99537 0.00463035), ppl = 9.99717
iteration 13, lambda = (0.996469 0.00353118), ppl = 9.99456
19 non-oov words, best lambda (0.997298 0.00270239)
pairwise cumulative lambda (1 0.00270239)


## Interpolation: The Secret Sauce of ASR  

compute-best-mix outputs a number like 0.95.   
This means: P(final_word) = 0.95 * P(domain_LM) + 0.05 * P(general_LM)  

**Why do we do this?**  
If you build an LM for Medical Transcription, it will be great at words like "acetaminophen", but terrible at common words like "going to". By interpolating a General English LM (trained on Wikipedia) with your Medical LM, you get the best of both worlds. SRILM's compute-best-mix uses Expectation-Maximization (EM) on a tuning set to find the exact mathematical multiplier that minimizes Perplexity.  


## Neural LM (PyTorch LSTM on GPU)

Karpathy's numpy RNN is great for math, but since we have an RTX 3090, let's train a PyTorch LSTM on the GPU to show modern workflows.  



In [85]:
!nvidia-smi

Fri May  1 17:08:14 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090 Ti     Off | 00000000:01:00.0  On |                  Off |
| 31%   55C    P2             121W / 480W |    755MiB / 24564MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
#!pip install -q torch torchvision

## Preparing Bigger Dataset  

myPOS ကိုပဲ သုံးကြရအောင်။  

```
ye@lst-hpc3090:/mnt/disk1/ye/exp/LM-Tutorial/data$ cp /home/ye/exp/SCRDR_tutorial/tokenizer/data/mypos_v3.word .
ye@lst-hpc3090:/mnt/disk1/ye/exp/LM-Tutorial/data$ cp /home/ye/exp/SCRDR_tutorial/tokenizer/data/10k_test.txt .
ye@lst-hpc3090:/mnt/disk1/ye/exp/LM-Tutorial/data$ ls
10k_test.txt  mypos_v3.word
ye@lst-hpc3090:/mnt/disk1/ye/exp/LM-Tutorial/data$
```


In [89]:
%cd /mnt/disk1/ye/exp/LM-Tutorial/data

/mnt/disk1/ye/exp/LM-Tutorial/data


In [90]:
!wc *

  10000  117857 1686379 10k_test.txt
  43196  564517 7518832 mypos_v3.word
  53196  682374 9205211 total


In [91]:
!head mypos_v3.word

၁၉၆၂ ခုနှစ် ခန့်မှန်း သန်းခေါင်စာရင်း အရ လူဦးရေ ၁၁၅၉၃၁ ယောက် ရှိ သည် ။
လူ တိုင်း တွင် သင့်မြတ် လျော်ကန် စွာ ကန့်သတ် ထား သည့် အလုပ် လုပ် ချိန် အပြင် ၊ လစာ နှင့်တကွ အခါ ကာလ အားလျော်စွာ သတ်မှတ် ထား သည့် အလုပ် အားလပ်ရက် များ ပါဝင် သည့် အနားယူခွင့် နှင့် အားလပ်ခွင့် ခံစားပိုင်ခွင့် ရှိ သည် ။
ဤ နည်း ကို စစ်ယူ သော နည်း ဟု ခေါ် သည် ။
စာပြန်ပွဲ ဆို တာ က အာဂုံဆောင် အလွတ်ကျက် ထား တဲ့ ပိဋကတ်သုံးပုံ စာပေ တွေ ကို စာစစ် သံဃာတော်ကြီး တွေ ရဲ့ ရှေ့ မှာ အလွတ် ပြန် ပြီး ရွတ်ပြ ရ တာ ပေါ့ ။
ဒီ မှာ ကျွန်တော့် သက်သေခံကတ် ပါ ။
၂ဝ ရာစု မြန်မာ့ သမိုင်း သန်းဝင်းလှိုင် ၊ ၂ဝဝ၉ ခု ၊ မေ လ ၊ ကံကော်ဝတ်ရည် စာပေ ။
ကျွန်တော် မျက်မှန် တစ် လက် လုပ် ချင် ပါ တယ် ။
ကျွန်တော် တို့ က ဒီ အမှု ရဲ့ ကြံရာပါ ကို ဖမ်းမိ ဖို့ ကြိုးစား ခဲ့ တယ် ။
ကလေး မီးဖွား ဖို့ ခန့်မှန်း ရက် က ဘယ်တော့ ပါ လဲ ။
အရိုးရှင်းဆုံး ကာဗိုဟိုက်ဒရိတ် မှာ ဂလူးကို့စ် ၊ ဂလက်တို့စ် ၊ ဖရပ်တို့စ် စသည့် မိုနိုဆက်ကရိုက် များ ဖြစ် သည် ။


In [92]:
!head 10k_test.txt

၁၉၆၂ ခုနှစ် ခန့်မှန်း သန်းခေါင်စာရင်း အရ လူဦးရေ ၁၁၅၉၃၁ ယောက် ရှိ သည်
လူ တိုင်း တွင် သင့်မြတ် လျော်ကန် စွာ ကန့်သတ် ထား သည့် အလုပ် လုပ် ချိန် အပြင် လစာ နှင့်တကွ အခါ ကာလ အားလျော်စွာ သတ်မှတ် ထား သည့် အလုပ် အားလပ်ရက် များ ပါဝင် သည့် အနားယူခွင့် နှင့် အားလပ်ခွင့် ခံစားပိုင်ခွင့် ရှိ သည်
ဤ နည်း ကို စစ်ယူ သော နည်း ဟု ခေါ် သည်
စာပြန်ပွဲ ဆို တာ က အာဂုံဆောင် အလွတ်ကျက် ထား တဲ့ ပိဋကတ်သုံးပုံ စာပေ တွေ ကို စာစစ် သံဃာတော်ကြီး တွေ ရဲ့ ရှေ့ မှာ အလွတ် ပြန် ပြီး ရွတ်ပြ ရ တာ ပေါ့
ဒီ မှာ ကျွန်တော့် သက်သေခံကတ် ပါ
၂ဝ ရာစု မြန်မာ့ သမိုင်း သန်းဝင်းလှိုင် ၂ဝဝ၉ ခု မေ လ ကံကော်ဝတ်ရည် စာပေ
ကျွန်တော် မျက်မှန် တစ် လက် လုပ် ချင် ပါ တယ်
ကျွန်တော် တို့ က ဒီ အမှု ရဲ့ ကြံရာပါ ကို ဖမ်းမိ ဖို့ ကြိုးစား ခဲ့ တယ်
ကလေး မီးဖွား ဖို့ ခန့်မှန်း ရက် က ဘယ်တော့ ပါ လဲ
အရိုးရှင်းဆုံး ကာဗိုဟိုက်ဒရိတ် မှာ ဂလူးကို့စ် ဂလက်တို့စ် ဖရပ်တို့စ် စသည့် မိုနိုဆက်ကရိုက် များ ဖြစ် သည်


## Open-test  

```
ye@lst-hpc3090:/mnt/disk1/ye/exp/LM-Tutorial/data$ cp /home/ye/exp/SCRDR_tutorial/tagger/pos_data/otest.word .
ye@lst-hpc3090:/mnt/disk1/ye/exp/LM-Tutorial/data$  
```

In [97]:
!wc /home/ye/exp/SCRDR_tutorial/tagger/pos_data/otest.word

  1000  13468 180788 /home/ye/exp/SCRDR_tutorial/tagger/pos_data/otest.word


In [98]:
!head /home/ye/exp/SCRDR_tutorial/tagger/pos_data/otest.word

တစ် ကိုက် ကို ဝမ် ခုနှစ်ထောင် ပါ ။
မနှစ် က သူ ကျွန်မ ကို သင် ပေး တယ် ။
ကျွန်တော့် ခုံ သွား ရှာ မလို့ ။
အတန်း စ တာ ကြာ ပြီ လား ။
ဆေး နည်းနည်း စား လိုက် ၊ သုံး လေး ရက် လောက် အနားယူ လိုက် ရင် ပျောက် သွား မှာ ပါ ။
အေးချမ်း မှု နဲ့ စည်းကမ်း ကို တည်မြဲ အောင် ထိန်းသိမ်း သည် ။
ဇွန်း ကို လိုအပ် တယ် ။
ဘွဲ့ ရ ရင် ဘာ လုပ် မ လို့ လဲ ။
ကျွန်တော် ချောင်းဆိုး ခြင်း အတွက် တစ် ခု ခု လို ချင် တယ် ။
အသီးအနှံ တို့ မှ လွဲ လျှင် လူ တို့ ၏ အဓိက အစားအစာ မှာ ငါး ဖြစ် သည် ။


## Simple Data Cleaning

LM မဆောက်ခင် ပုဒ်ထီး၊ ပုဒ်မတွေ ဖြုတ်ရအောင်။  
တကယ်ကတော့ ရှင်းမယ်ဆိုရင် တခြား punctuation mark တွေ အင်္ဂလိပ်စာတွေပါ ရှင်းသင့်တယ်။  



In [103]:
!cat /mnt/disk1/ye/exp/LM-Tutorial/data/clean_text.py

import argparse
import string

def remove_punctuation(input_path, output_path):
    # Define English punctuation from the string module
    english_punct = string.punctuation
    
    # Define Burmese specific punctuation
    burmese_punct = "၊။"
    
    # Combine them into one set of characters to remove
    all_to_remove = english_punct + burmese_punct
    
    # Create a translation table for fast processing
    # This maps every character in 'all_to_remove' to None
    table = str.maketrans('', '', all_to_remove)

    try:
        with open(input_path, 'r', encoding='utf-8') as infile:
            with open(output_path, 'w', encoding='utf-8') as outfile:
                for line in infile:
                    # Apply the translation table to each line
                    clean_line = line.translate(table)
                    outfile.write(clean_line)
        
        print(f"Success! Cleaned text saved to: {output_path}")
        
    except FileNotFoundError:
        print(f"Erro

In [104]:
%cd /mnt/disk1/ye/exp/LM-Tutorial/data

/mnt/disk1/ye/exp/LM-Tutorial/data


In [105]:
!python ./clean_text.py --input ./mypos_v3.word --output ./mypos_v3.word.clean

Success! Cleaned text saved to: ./mypos_v3.word.clean


In [106]:
!head ./mypos_v3.word.clean

၁၉၆၂ ခုနှစ် ခန့်မှန်း သန်းခေါင်စာရင်း အရ လူဦးရေ ၁၁၅၉၃၁ ယောက် ရှိ သည် 
လူ တိုင်း တွင် သင့်မြတ် လျော်ကန် စွာ ကန့်သတ် ထား သည့် အလုပ် လုပ် ချိန် အပြင်  လစာ နှင့်တကွ အခါ ကာလ အားလျော်စွာ သတ်မှတ် ထား သည့် အလုပ် အားလပ်ရက် များ ပါဝင် သည့် အနားယူခွင့် နှင့် အားလပ်ခွင့် ခံစားပိုင်ခွင့် ရှိ သည် 
ဤ နည်း ကို စစ်ယူ သော နည်း ဟု ခေါ် သည် 
စာပြန်ပွဲ ဆို တာ က အာဂုံဆောင် အလွတ်ကျက် ထား တဲ့ ပိဋကတ်သုံးပုံ စာပေ တွေ ကို စာစစ် သံဃာတော်ကြီး တွေ ရဲ့ ရှေ့ မှာ အလွတ် ပြန် ပြီး ရွတ်ပြ ရ တာ ပေါ့ 
ဒီ မှာ ကျွန်တော့် သက်သေခံကတ် ပါ 
၂ဝ ရာစု မြန်မာ့ သမိုင်း သန်းဝင်းလှိုင်  ၂ဝဝ၉ ခု  မေ လ  ကံကော်ဝတ်ရည် စာပေ 
ကျွန်တော် မျက်မှန် တစ် လက် လုပ် ချင် ပါ တယ် 
ကျွန်တော် တို့ က ဒီ အမှု ရဲ့ ကြံရာပါ ကို ဖမ်းမိ ဖို့ ကြိုးစား ခဲ့ တယ် 
ကလေး မီးဖွား ဖို့ ခန့်မှန်း ရက် က ဘယ်တော့ ပါ လဲ 
အရိုးရှင်းဆုံး ကာဗိုဟိုက်ဒရိတ် မှာ ဂလူးကို့စ်  ဂလက်တို့စ်  ဖရပ်တို့စ် စသည့် မိုနိုဆက်ကရိုက် များ ဖြစ် သည် 


In [107]:
!python ./clean_text.py --input ./10k_test.txt --output ./10k_test.txt.clean

Success! Cleaned text saved to: ./10k_test.txt.clean


In [108]:
!head ./10k_test.txt.clean

၁၉၆၂ ခုနှစ် ခန့်မှန်း သန်းခေါင်စာရင်း အရ လူဦးရေ ၁၁၅၉၃၁ ယောက် ရှိ သည်
လူ တိုင်း တွင် သင့်မြတ် လျော်ကန် စွာ ကန့်သတ် ထား သည့် အလုပ် လုပ် ချိန် အပြင် လစာ နှင့်တကွ အခါ ကာလ အားလျော်စွာ သတ်မှတ် ထား သည့် အလုပ် အားလပ်ရက် များ ပါဝင် သည့် အနားယူခွင့် နှင့် အားလပ်ခွင့် ခံစားပိုင်ခွင့် ရှိ သည်
ဤ နည်း ကို စစ်ယူ သော နည်း ဟု ခေါ် သည်
စာပြန်ပွဲ ဆို တာ က အာဂုံဆောင် အလွတ်ကျက် ထား တဲ့ ပိဋကတ်သုံးပုံ စာပေ တွေ ကို စာစစ် သံဃာတော်ကြီး တွေ ရဲ့ ရှေ့ မှာ အလွတ် ပြန် ပြီး ရွတ်ပြ ရ တာ ပေါ့
ဒီ မှာ ကျွန်တော့် သက်သေခံကတ် ပါ
၂ဝ ရာစု မြန်မာ့ သမိုင်း သန်းဝင်းလှိုင် ၂ဝဝ၉ ခု မေ လ ကံကော်ဝတ်ရည် စာပေ
ကျွန်တော် မျက်မှန် တစ် လက် လုပ် ချင် ပါ တယ်
ကျွန်တော် တို့ က ဒီ အမှု ရဲ့ ကြံရာပါ ကို ဖမ်းမိ ဖို့ ကြိုးစား ခဲ့ တယ်
ကလေး မီးဖွား ဖို့ ခန့်မှန်း ရက် က ဘယ်တော့ ပါ လဲ
အရိုးရှင်းဆုံး ကာဗိုဟိုက်ဒရိတ် မှာ ဂလူးကို့စ် ဂလက်တို့စ် ဖရပ်တို့စ် စသည့် မိုနိုဆက်ကရိုက် များ ဖြစ် သည်


In [110]:
!python ./clean_text.py --input ./otest.word --output ./otest.word.clean

Success! Cleaned text saved to: ./otest.word.clean


In [113]:
!head ./otest.word.clean

တစ် ကိုက် ကို ဝမ် ခုနှစ်ထောင် ပါ 
မနှစ် က သူ ကျွန်မ ကို သင် ပေး တယ် 
ကျွန်တော့် ခုံ သွား ရှာ မလို့ 
အတန်း စ တာ ကြာ ပြီ လား 
ဆေး နည်းနည်း စား လိုက်  သုံး လေး ရက် လောက် အနားယူ လိုက် ရင် ပျောက် သွား မှာ ပါ 
အေးချမ်း မှု နဲ့ စည်းကမ်း ကို တည်မြဲ အောင် ထိန်းသိမ်း သည် 
ဇွန်း ကို လိုအပ် တယ် 
ဘွဲ့ ရ ရင် ဘာ လုပ် မ လို့ လဲ 
ကျွန်တော် ချောင်းဆိုး ခြင်း အတွက် တစ် ခု ခု လို ချင် တယ် 
အသီးအနှံ တို့ မှ လွဲ လျှင် လူ တို့ ၏ အဓိက အစားအစာ မှာ ငါး ဖြစ် သည် 


## LSTM Based LM

Code ကို  လေ့လာကြည့်ပါ။  

In [72]:
%cd /mnt/disk1/ye/exp/LM-Tutorial/lstm

/mnt/disk1/ye/exp/LM-Tutorial/lstm


In [73]:
!cat lstm_lm.py

#!/usr/bin/env python3
"""
A complete, standalone LSTM Language Model with Train, Test, and Generate modes.
Specifically designed to handle OOVs safely and support Character-level modeling (ideal for Myanmar).
"""

import os
import math
import json
import argparse
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ==========================================
# 1. DATA HANDLING & VOCABULARY
# ==========================================
class Vocabulary:
    def __init__(self):
        self.token2idx = {"<pad>": 0, "<unk>": 1, "<s>": 2, "</s>": 3}
        self.idx2token = {0: "<pad>", 1: "<unk>", 2: "<s>", 3: "</s>"}
    
    def build(self, tokens):
        """Build vocab from a list of tokens (words or characters)"""
        for token in set(tokens):
            if token not in self.token2idx:
                self.token2idx[token] = len(self.token2idx)
                self.idx2token[len(self.idx2token)] = token
        return self

    def __len__(self)

In [74]:
!python ./lstm_lm.py --help

usage: lstm_lm.py [-h] --mode {train,test,generate} [--train_file TRAIN_FILE]
                  [--test_file TEST_FILE] [--model_path MODEL_PATH]
                  [--token_level {word,char}] [--embed_dim EMBED_DIM]
                  [--hidden_dim HIDDEN_DIM] [--num_layers NUM_LAYERS]
                  [--seq_len SEQ_LEN] [--epochs EPOCHS]
                  [--batch_size BATCH_SIZE] [--lr LR] [--prompt PROMPT]
                  [--gen_length GEN_LENGTH] [--temperature TEMPERATURE]

LSTM Language Model Trainer/Evaluator/Generator

options:
  -h, --help            show this help message and exit
  --mode {train,test,generate}
                        What to do?
  --train_file TRAIN_FILE
                        Path to training text file
  --test_file TEST_FILE
                        Path to test text file
  --model_path MODEL_PATH
                        Path to save/load model
  --token_level {word,char}
                        'word' for English, 'char' for Myanmar or unsegmented
    

## Shell Scripts  

Training, Evaluation နဲ့ Text generation တွေအတွက် shell script တွေလည်း ပြင်ဆင်ထားတယ်။  

In [75]:
%cd /mnt/disk1/ye/exp/LM-Tutorial/lstm

/mnt/disk1/ye/exp/LM-Tutorial/lstm


In [76]:
!cat train.sh

#!/bin/bash

# Train (takes ~10 seconds on your RTX 3090)
time python lstm_lm.py --mode train \
                  --train_file ../data/mypos_v3.word.clean \
                  --model_path myanmar_word_lm.pt \
                  --token_level word \
                  --epochs 30 \
                  --seq_len 50


In [118]:
!./train.sh

Using device: cuda
Reading training data from ../data/mypos_v3.word.clean (Level: word)...
Building vocabulary...
Vocabulary size: 24720 tokens
Starting training for 30 epochs...
Epoch   5/30 | Loss: 1.8638
Epoch  10/30 | Loss: 1.5201
Epoch  15/30 | Loss: 1.3898
Epoch  20/30 | Loss: 1.3186
Epoch  25/30 | Loss: 1.2714
Epoch  30/30 | Loss: 1.2373
Saving model to myanmar_word_lm.pt...
Training complete!

real	85m30.684s
user	85m16.968s
sys	0m20.835s


## Check GPU Status During LSTM-based LM Training  

```
ye@lst-hpc3090:/mnt/disk1/ye/exp$ nvidia-smi
Fri May  1 18:08:58 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090 Ti     Off | 00000000:01:00.0  On |                  Off |
| 50%   76C    P2             356W / 480W |   2434MiB / 24564MiB |     87%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+
                                                                                         
+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|
|    0   N/A  N/A     51185      G   /usr/lib/xorg/Xorg                          195MiB |
|    0   N/A  N/A     51389      G   /usr/bin/gnome-shell                         64MiB |
|    0   N/A  N/A     52003      G   /usr/libexec/xdg-desktop-portal-gnome         6MiB |
|    0   N/A  N/A     52268      C   /usr/NX/bin/nxnode.bin                      276MiB |
|    0   N/A  N/A    300604      G   ...1/usr/bin/snapd-desktop-integration       16MiB |
|    0   N/A  N/A    479401      G   /usr/bin/nautilus                            32MiB |
|    0   N/A  N/A   1780390      G   ...ads/tsetup.5.10.7/Telegram/Telegram       40MiB |
|    0   N/A  N/A   3125725      G   ...n=20260425-030030.652000-production       53MiB |
|    0   N/A  N/A   3138305      G   ...irefox/8191/usr/lib/firefox/firefox      137MiB |
|    0   N/A  N/A   3495846      C   /usr/bin/python3                            368MiB |
|    0   N/A  N/A   3536063      C   python                                     1192MiB |
+---------------------------------------------------------------------------------------+
```

In [119]:
!wc myanmar_word_lm.pt.vocab

     0  49440 854006 myanmar_word_lm.pt.vocab


In [120]:
# This slices the object and pipes it to a readable format
!jq 'to_entries | .[:100] | from_entries' myanmar_word_lm.pt.vocab

{
  "<pad>": 0,
  "<unk>": 1,
  "<s>": 2,
  "</s>": 3,
  "​ဒါ": 4,
  "Institutet": 5,
  "ပုစ္ဆာဝိသဇ္ဇနာ": 6,
  "မုတ်သုံ": 7,
  "အချိန်မလပ်": 8,
  "လွှမ်းခြုံ": 9,
  "မွေးဖွား": 10,
  "ရေးထိုး": 11,
  "နှစ်ဦးနှစ်ဖက်လုံး": 12,
  "ပြန်ချေပ": 13,
  "ခန့်": 14,
  "ငတ်မွတ်": 15,
  "ခရီး": 16,
  "ပင့်": 17,
  "မဟာဝိဇယ": 18,
  "Lies": 19,
  "ယောင်ဝါးဝါး": 20,
  "တောင်တွင်းရှင်မုန်ယို": 21,
  "မောတောင်": 22,
  "မုန်လာဥနီ": 23,
  "redgiant": 24,
  "အထူဆုံး": 25,
  "တွမ်": 26,
  "ဓါတုဗေဒ": 27,
  "၁၉၂၉၆": 28,
  "အယ်လိုရာ": 29,
  "နာကျင်မှု": 30,
  "ကွဲပြား": 31,
  "အဝှမ်း": 32,
  "ဆိုးဆိုးဝါးဝါး": 33,
  "ဗိဆေးလိယပ်": 34,
  "ခေါင်းကိုက်ရောဂါ": 35,
  "သရ": 36,
  "မိုင်းသောက်": 37,
  "userpage": 38,
  "လူပြိန်း": 39,
  "အေးတိအေးစက်": 40,
  "ရန်သူ": 41,
  "BIA": 42,
  "လက္ခဏာ": 43,
  "ချွတ်ယွင်းချက်": 44,
  "‌တောင်ကုန်း": 45,
  "ပူဖောင်း": 46,
  "‌‌ကောင်မလေး": 47,
  "ခရီးဆောင်အိတ်": 48,
  "ကောက်ဆန်": 49,
  "မူပိုင်": 50,
  "language": 51,
  "အကြင်နာ": 52,
  "အယူဝါဒခြား": 53,
  "ကမ္ဘာလှည့်ခရီးသွားလုပ်င

jq command က စက်ထဲမှာ မရှိရင်တော့ အောက်ပါလိုမျိုး fold command နဲ့ head command နဲ့လည်း တွဲကြည့်လို့ ရလိမ့်မယ်။  

In [122]:
# Wrap the text at 80 characters and show the first 20 lines
!fold -s -w 80 myanmar_word_lm.pt.vocab | head -n 20

{"<pad>": 0, "<unk>": 1, "<s>": 2, "</s>": 3, "​ဒါ": 4, "Institutet": 5, 
"ပုစ္ဆာဝိသဇ္ဇနာ": 6, "မုတ်သုံ": 7, 
"အချိန်မလပ်": 8, "လွှမ်းခြုံ": 9, 
"မွေးဖွား": 10, "ရေးထိုး": 11, 
"နှစ်ဦးနှစ်ဖက်လုံး": 12, 
"ပြန်ချေပ": 13, "ခန့်": 14, "ငတ်မွတ်": 
15, "ခရီး": 16, "ပင့်": 17, "မဟာဝိဇယ": 18, 
"Lies": 19, "ယောင်ဝါးဝါး": 20, 
"တောင်တွင်းရှင်မုန်ယို": 21, 
"မောတောင်": 22, "မုန်လာဥနီ": 23, "redgiant": 
24, "အထူဆုံး": 25, "တွမ်": 26, "ဓါတုဗေဒ": 
27, "၁၉၂၉၆": 28, "အယ်လိုရာ": 29, 
"နာကျင်မှု": 30, "ကွဲပြား": 31, 
"အဝှမ်း": 32, "ဆိုးဆိုးဝါးဝါး": 33, 
"ဗိဆေးလိယပ်": 34, 
"ခေါင်းကိုက်ရောဂါ": 35, "သရ": 36, 
"မိုင်းသောက်": 37, "userpage": 38, 
"လူပြိန်း": 39, "အေးတိအေးစက်": 40, 
"ရန်သူ": 41, "BIA": 42, "လက္ခဏာ": 43, 
"ချွတ်ယွင်းချက်": 44, 
fold: write error: Broken pipe


## Evaluation on LSTM LM

In [78]:
!cat eval.sh

#!/bin/bash

time python lstm_lm.py --mode test \
                  --test_file ../data/otest.word.clean \
                  --model_path ./myanmar_word_lm.pt \
                  --token_level word \
                  --seq_len 50


In [79]:
!./eval.sh

Using device: cuda
Loading model from ./myanmar_word_lm.pt...
Reading test data from ../data/otest.word.clean...
Evaluating...
Test Results -> Tokens: 607700, Avg NLL: 0.7512, PPL: 2.12

real	0m22.087s
user	0m22.282s
sys	0m0.642s


တကယ်က train လုပ်ထားတာက ပုဒ်ထီး၊ ပုဒ်မတွေကို ဖြုတ်ထားတဲ့ cleaning လုပ်ထားတဲ့ open test ဒေတာနဲ့ evaluation လုပ်ရမှာပါ။   
eval.sh ကို update လုပ်လိုက်ပြီး နောက်တစ်ခေါက် evaluation လုပ်ကြည့်ရအောင်။  

In [84]:
!cat ./eval.sh

#!/bin/bash

time python lstm_lm.py --mode test \
                  --test_file ../data/otest.word \
                  --model_path ./myanmar_word_lm.pt \
                  --token_level word \
                  --seq_len 50


In [83]:
!./eval.sh

Using device: cuda
Loading model from ./myanmar_word_lm.pt...
Reading test data from ../data/otest.word...
Evaluating...
Test Results -> Tokens: 670900, Avg NLL: 7.2281, PPL: 1377.59

real	0m24.831s
user	0m25.147s
sys	0m0.572s


သိစေချင်တာက test ဒေတာကလည်း ဘယ်လောက် အရေးကြီးတယ်ဆိုတာ။ ပြီးတော့ training လုပ်ထားတဲ့ ဒေတာ၊ ဒိုမိန်း စတာတွေက ဘယ်လောက်ထိ PPL ရလဒ်ကို အပြောင်းအလဲ ဖြစ်နိုင်သလဲ ဆိုတာကိုပါ။  

## Text Generation


In [85]:
!cat ./gen.sh

python lstm_lm.py --mode generate \
                  --model_path myanmar_word_lm.pt \
                  --token_level word \
                  --prompt "ပြည်ထောင်စု" \
                  --gen_length 200 \
                  --temperature 0.7


In [86]:
!./gen.sh

Using device: cuda
Loading model from myanmar_word_lm.pt...

Prompt: ပြည်ထောင်စု
Generating: 
ပြည်ထောင်စု ဝတ္ထု ကို အစောဆုံး သော မြို့ ဖြစ် သည် မှတ်ပုံတင် ချင် တာ ဘယ် မှာ လဲ ဆို တာ ပြ ပေး ပါ ခရစ် အသုံးပြု ချိန် စနစ် များ အရ မြန်မာ့ ယဉ်ကျေး မှု ကို ခံစား သည့် အခါ အခါ သိရှိ ရာ၌ သိပ္ပံနည်း များ အချင်းချင်း နှင့် အပင် များ တို့ သည် နေ အဖွဲ့အစည်း အတွင်း တွင် ရှိ ပြီး မြောက် ပိုင်း တွင် မဇ္ဈိမ ဒေသ ကြီး များ မှ ၎င်း ကို ကမ္ဘာ နိုင်ငံ ရှိ နိုင်ငံ များ တွင် စီးပွား ရေး ဖွံ့ဖြိုး လုပ်ဆောင် မှု များ ရှိ သည် အထူး တိုင်းတာ မှု ကုမ္ပဏီ ဝင်ငွေခွန် မ ရ ရန် အတွက် စီစဉ် ထား သော အဖွဲ့ ဝင် ဖြစ် သည့် မြစေတီ ဖောင့် ကို စတင် ချီးမြှင့် သည် တကယ့်ကို ကောင်း တာ ပဲ ကျွန်မ ရဲ့ အမှား ကြောင့် သူ နဲ့ သူ ဒီ နေ့ ဆွမ်းချိုင့် ပို့ ဖို့ ဆွမ်း ဟင်း ချက် နေ ရစ် တယ် တကယ် လို့ ဘာ လုပ် ချင် လဲ မြန်မာ ငွေ ၃ ခု က အဆို အက အစီအစဉ် ရှိ ပါ သလား ခင်ဗျား က ဖော်ရွေ ပြီး တော့ ကူညီ ပါ ကျွန်တော် ဆေး ကူညီ မယ် လို့ စဉ်းစား နေ တာ ခင်ဗျား စိတ်ကူးစိတ်သန်း ကောင်း တယ် ကား ပြင် ဖို့ စောင့် ရ မယ် မိဘ တွေ က အသက်ကြီး တာ နဲ့ သူဌေး ပဲ တစ် နာရီ လောက်

**ဆရာ temp နဲ့ length ကို အပြောင်းအလဲ လုပ်လိုက်တယ်။ ထပ် generation လုပ်ခိုင်းကြမယ်။**  

In [87]:
!./gen.sh

Using device: cuda
Loading model from myanmar_word_lm.pt...

Prompt: ပြည်ထောင်စု
Generating: 
ပြည်ထောင်စု ကြံ့ခိုင် ရေး နှင့် ဖွံ့ဖြိုး ရေး အသင်း ၏ ဥက္ကဋ္ဌ ဖြစ် ခဲ့ သည် အနီးဆုံး ဘဏ် က ဘယ် အချိန် ထွက် လဲ အကျပ်အတည်း စည်းကမ်း အားလုံး ဆိုး တယ် ခင်ဗျား တို့ ကုန်ပစ္စည်း က ဘာ အားသာချက် တွေ ရှိ ပါ သလဲ ခင်ဗျား မှာ ကလေး တစ်ယောက်စာ ပြင် တဲ့ ပွဲ ရှိ လား ကပ်ကြေး ကျွန်တော် တို့ ဘယ်လောက် ကြာကြာ စောင့် ရ မလဲ


## Evaluation with Closed-Test Data

Closed-test ဒေတာ ဒါပေမဲ့ စာကြောင်းရေက များတယ်။ အဲဒီ ဒေတာနဲ့ LSTM LM ကို evaluation လုပ်ကြည့်ပြီး PPL က ဘယ်လို အပြောင်းအလဲ ဖြစ်သလဲ ဆိုတာကို လေ့လာကြည့်ကြရအောင်။  

In [136]:
!wc ../data/10k_test.txt.clean

  10000  117817 1686267 ../data/10k_test.txt.clean


In [137]:
!cat ./eval-closed.sh

#!/bin/bash

time python lstm_lm.py --mode test \
                  --test_file ../data/10k_test.txt.clean \
                  --model_path ./myanmar_word_lm.pt \
                  --token_level word \
                  --seq_len 50


In [139]:
!./eval-closed.sh

Using device: cuda
Loading model from ./myanmar_word_lm.pt...
Reading test data from ../data/10k_test.txt.clean...
Evaluating...
Test Results -> Tokens: 5888350, Avg NLL: 0.7543, PPL: 2.13

real	2m26.987s
user	2m27.241s
sys	0m0.736s


Closed-test နဲ့ evaluation လုပ်တာလည်း ရလဒ်ကောင်းပါတယ်။  
LM ပဲ ဖြစ်ဖြစ်၊ ဘယ်မော်ဒယ်ပဲ ဖြစ်ဖြစ် နှစ်မျိုးစလုံးနဲ့ စစ်သင့်ပါတယ်။  
ဒါပေမဲ့ ပိုအရေးကြီးတာက open test ဒေတာရဲ့ ရလဒ်ပါ။  

## Transformer LM (HuggingFace on GPU)

We use GPT-2. We explicitly cast it to bfloat16 which the RTX 3090 supports natively, doubling throughput and halving VRAM usage with zero accuracy loss.  

ဒီ ဥပမာ code ကတော့ HuggingFace ကနေ ရှိပြီးသား မော်ဒယ်ကို ယူသုံးပြီး datasets ဆိုတဲ့ Python library ကို import လုပ်ပြီး publicly share လုပ်ပေးထားတဲ့ wiki dataset ကိုသုံးပြီး evaluation လုပ်ပြထားတာပါ။ အင်္ဂလိပ်စာတို့လို ဂျာမန်တို့လို့ ဂျပန်၊ တရုတ် စတဲ့ ဘာသာစကားတွေအတွက်က ဒေတာအများကြီးသုံးပြီး ဆောက်ထားတဲ့ LM တွေက အဆင်သင့်ရှိတာမို့ တော်တော်လေး အဆင်ပြေပါတယ်။ ဒါပေမဲ့ ဆရာတို့ မြန်မစာအတွက်က အခြေခံအားဖြင့် အရမ်းကြီးအဆင်မပြေလို့ ကိုယ်လုပ်ချင်တဲ့ ဒိုမိန်းအတွက် LM ကို သပ်သပ်ဆောက်ပြီးပဲ သုံးကြရတာ များပါလိမ့်မယ်။  

In [140]:
#!pip install -q transformers datasets accelerate

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Load in bfloat16 to maximize RTX 3090 efficiency
model_gpt2 = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16).to(device)

# Load a real test set (WikiText-2) to get meaningful PPL numbers
test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
encodings = tokenizer("\n\n".join(test["text"]), return_tensors="pt").to(device)

# Strided Perplexity Evaluation
from tqdm.auto import tqdm

max_length = model_gpt2.config.n_positions
stride = 512
seq_len = encodings.input_ids.size(1)

nll_sum, n_tokens = 0.0, 0
prev_end_loc = 0

print("Evaluating GPT-2 (bf16)...")
with torch.no_grad():
    for begin_loc in tqdm(range(0, seq_len, stride)):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc

        input_ids = encodings.input_ids[:, begin_loc:end_loc]
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        outputs = model_gpt2(input_ids, labels=target_ids)
        neg_log_likelihood = outputs.loss.item()

        num_valid_tokens = (target_ids != -100).sum().item()
        num_loss_tokens = num_valid_tokens - target_ids.size(0) # HF shifts labels internally

        nll_sum += neg_log_likelihood * num_loss_tokens
        n_tokens += num_loss_tokens
        prev_end_loc = end_loc
        
        if end_loc == seq_len: break

ppl_gpt2, ent_gpt2 = compute_ppl_and_entropy(nll_sum, n_tokens)
print(f"\nGPT-2 Test -> PPL: {ppl_gpt2:.2f}, Entropy: {ent_gpt2:.2f} nats")

/home/ye/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-01 21:50:18.018010: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-01 21:50:20.632506: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777647021.674408 3495846 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777647022.019741 3495846 cuda_blas.cc:1407] Unable to regi

Evaluating GPT-2 (bf16)...


100%|█████████████████████████████████████████████████████████████████▊| 560/562 [00:07<00:00, 75.81it/s]


GPT-2 Test -> PPL: 25.41, Entropy: 3.24 nats


## Transformer LM (with Myanmar language)  

တကယ်ကတော့ LSTM နဲ Transformer မော်ဒယ်အကြားမှာ မတူတာတွေ အများကြီးပါပဲ။ Architecture ပိုင်းတင်မကပဲ Training လုပ်တဲ့ နည်းလမ်းတွေမှာလည်း အများကြီး အပြောင်းအလဲတွေရှိပါတယ်။ ဆရာ အခုသုံးနေတဲ့ GPU စက်က RTX 3090 (24GB VRAM) ရှိတာမို့ mixed-precision training (bf16) ဆိုတာကို natively support လုပ်တာမို့လို့ Transformer မော်ဒယ် training time က အရမ်းမြန်သွားပါလိမ့်မယ်။  

ပြီးတော့ Transformer ပေါ်လာပြီး နောက်ပိုင်းမှာ မော်ဒယ်ကို အစအဆုံး training လုပ်တာထက် pretrained လုပ်ထားတဲ့ အဆင်ပြေတဲ့ မော်ဒယ်ကိုပဲ အခြေခံပြီးတော့ Finetuning လုပ်ယူကြတာက များပါတယ်။ ဘာကြောင့်လဲ ဆိုတော့ Transformer ကို အခြေခံတဲ့ LM ကောင်းကောင်းဆောက်ဖို့အတွက်ဆိုရင် monolingual data ကိုလည်း GB အများကြီး ပြင်ထားဖို့လိုအပ်ပြီး GPU လည်း အများကြီးသုံးကြရတာမို့ပါ။   

ဒီ LM Tutorial မှာတော့ pretrained model XGLM ကိုပဲ ကိုယ့် မြန်မာစာဒေတာနဲ့ finetune လုပ်ပြီး weight တွေကို learn လုပ်ယူသွားပါမယ်။  
XGLM က GPT-2 မဟုတ်ဘူးနော်။ သူက Meta ရဲ့ multilingual generative model ပါ။ သူက "GPT-style" Autoregressive အာခီပါ။ ပြီးတော့ မြန်မာစာအတွက် အဆင်ပြေတယ်။ သူ့ vocab size က 256k SentencePiece ပါ။    

In [142]:
%cd /mnt/disk1/ye/exp/LM-Tutorial/transformer

/mnt/disk1/ye/exp/LM-Tutorial/transformer


In [151]:
%cat ./transformer_lm.py

#!/usr/bin/env python3
"""
Transformer-based Language Model using Hugging Face.
Defaults to fine-tuning XGLM (a GPT-style autoregressive model). 
Note: Standard GPT-2 is NOT used by default because its tokenizer lacks Myanmar 
subwords, forcing it to split Myanmar text into inefficient UTF-8 bytes. 
XGLM uses a 256k SentencePiece vocabulary that natively includes Myanmar script.
"""

import os
import math
import argparse
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset

# ==========================================
# 1. DATA UTILITIES
# ==========================================
def load_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

def tokenize_dataset(lines, tokenizer, max_length):
    """Safely tokenizes and chunks te

## Finetuning  

တကယ်က အထက်ပါ warning message က HuggingFace library တွေကို Jupyter notebook အတွင်းက ခေါ်သုံးရင် ပေးနေကြ warning ပါ။ Multiprocessing လုပ်ရင် တွေ့ရတဲ့ message မျိုးပါ။ ဘာကြောင့်လဲ ဆိုတော့ tokenizer library က text processing speed မြန်ဆန်အောင်လို့ multiple CPU တွေသုံးမှာမို့လို့ပါ။  

Finetuning သို့မဟုတ် training တမျိုးလုပ်မယ့် script က အောက်ပါအတိုင်းပါ။  

In [89]:
%cd /mnt/disk1/ye/exp/LM-Tutorial/transformer

/mnt/disk1/ye/exp/LM-Tutorial/transformer


In [91]:
!cat train.sh

#!/bin/bash
# Fine-tuning XGLM on your data
# bf16 is enabled automatically in the python script for your RTX 3090

time python transformer_lm.py --mode train \
                  --train_file ../data/mypos_v3.word.clean \
                  --model_path ./myanmar_xglm_word \
                  --seq_len 128 \
                  --epochs 3 \
                  --batch_size 8


အောက်ပါလိုမျိုး warning ကို ပျောက်အောင် လုပ်ရအောင်။  

 huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...  

To disable this warning, you can either:
- Avoid using `tokenizers` before the fork if possible
- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false) 

In [29]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [152]:
!./train.sh

2026-05-01 22:56:21.964239: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-01 22:56:21.977467: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777650981.993242 3555120 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777650981.998437 3555120 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777650982.011564 3555120 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Evaluation with Closed Test Data  

အရင်ဆုံး finetuned လုပ်ထားတဲ့ မော်ဒယ်ကို closed test ဒေတာနဲ့ပဲ evaluation လုပ်ကြည့်ရအောင်။  

In [155]:
!./eval-closed.sh

2026-05-01 23:06:26.587383: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-01 23:06:26.610502: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777651586.636960 3555621 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777651586.645750 3555621 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777651586.668323 3555621 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Evaluation with Open Test Data  

ဒီတခါတော့ ပိုအရေးကြီးတဲ့ open test ဒေတာနဲ့ finetuning လုပ်ကြည့်ပါမယ်။  

In [156]:
!./eval.sh

2026-05-01 23:08:23.695043: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-01 23:08:23.709772: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777651703.727214 3555905 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777651703.732778 3555905 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777651703.747078 3555905 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Text Generation with XGLM Finetuned LM

ရလဒ်ကတော့ LSTM လောက် မကောင်းတာကို တွေ့ရလိမ့်မယ်။  
Text generation လုပ်ကြည့်ပြီး finetuned model ကို လေ့လာရအောင်။  

In [92]:
!cat gen.sh

#!/bin/bash
# Added --top_k and --top_p (Nucleus sampling) - standard practices for modern LLMs

python transformer_lm.py --mode generate \
                  --model_path ./myanmar_xglm_word \
                  --prompt "ပြည်ထောင်စု" \
                  --gen_length 200 \
                  --temperature 0.7 \
                  --top_k 50 \
                  --top_p 0.95


In [93]:
!./gen.sh

2026-05-02 15:09:23.666810: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 15:09:23.676116: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777709363.686797 3622283 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777709363.690333 3622283 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777709363.699245 3622283 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

**နောက်တစ်ခေါက် ထပ် generate လုပ်ခိုင်းကြည့်မယ်**

In [160]:
!./gen.sh

2026-05-01 23:14:07.843582: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-01 23:14:07.856815: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777652047.872796 3557020 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777652047.877939 3557020 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777652047.891292 3557020 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

**နောက်ထပ် တစ်ကြိမ် ထပ် generate လုပ်ခိုင်းကြည့်မယ်**

In [161]:
!./gen.sh

2026-05-01 23:15:07.914115: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-01 23:15:07.928626: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777652107.946286 3557224 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777652107.952125 3557224 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777652107.966249 3557224 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## LSTM vs XGLM

Your LSTM (~1 Million parameters): It is tiny. When you fed it your ~40,000-line dataset, it had more than enough capacity to simply memorize the entire text word-for-word. A PPL of 2.12 means the model is almost 100% certain of the next word. It’s not learning "Myanmar grammar"; it’s acting like a hard drive that just memorized the file. (If you test it on totally wild out-of-domain text, it will crash).
Your XGLM (564 Million parameters): It is a massive brain. When you fine-tune it for just 3 epochs on a relatively small dataset, the model barely has time to adjust its 564 million weights before the training stops. It underfit. It’s like asking a genius to read a 10-page pamphlet and then instantly change their whole speaking style—they will mostly stick to their original way of speaking.

reason 2: The "Tokenization Inequality" (Apples vs. Oranges)
You cannot directly compare the PPL of a Word-level LSTM with a Subword-level Transformer.

LSTM: Predicts 1 item out of 99 possible words.
XGLM: Predicts 1 item out of 256,000 possible subwords.
Because XGLM breaks words into smaller pieces (e.g., "ပြည်" might become 2 or 3 subwords), it has to make predictions more frequently, and the math naturally results in a higher PPL.
To compare them fairly, you would have to calculate Bits-Per-Character (BPC), where XGLM would likely win because it understands character context better.

## Fine-tuning XGLM with Optimized Parameters  

In [166]:
!cat ./train_optimize.sh

#!/bin/bash

# Fine-tuning XGLM with optimized parameters for smaller datasets
time python transformer_lm.py --mode train \
                  --train_file ../data/mypos_v3.word.clean \
                  --model_path ./myanmar_xglm_word_optimize \
                  --base_model facebook/xglm-564M \
                  --seq_len 512 \
                  --epochs 10 \
                  --batch_size 4 \
                  --lr 2e-5


In [167]:
!./train_optimize.sh

2026-05-01 23:28:41.634971: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-01 23:28:41.657260: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777652921.682828 3558656 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777652921.691453 3558656 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777652921.713381 3558656 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Evaluation with Original XGLM-564M LM

Finetuning က တကယ် အလုပ်လုပ်ရဲ့လား ဆိုတာကို သိနိုင်ဖို့ Meta ရဲ့ အော်ရဂျင်နယ် XGLM-564M language model ကို တိုက်ရိုက် evaluation လုပ်ကြည့်ကြရအောင်။ 10K (စာကြောင်း တစ်သောင်း) test ဖိုင်နဲ့ပဲလုပ်ကြည့်မယ်။   

In [168]:
!cat ./eval-base.sh

#!/bin/bash
# Evaluate the PRE-TRAINED base model WITHOUT fine-tuning

time python transformer_lm.py --mode test \
                  --test_file ../data/10k_test.txt.clean \
                  --model_path facebook/xglm-564M \
                  --base_model facebook/xglm-564M \
                  --seq_len 512 \
                  --stride 256


In [169]:
!./eval-base.sh

2026-05-02 00:00:11.337484: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 00:00:11.360414: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777654811.386470 3560106 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777654811.395229 3560106 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777654811.417614 3560106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Evaluation with Finetune Optimized Model  

နောက်ဆုံး epoch 10 ပြီးတော့ --seq_len 512 နဲ့ဆောက်ထားတဲ့ မော်ဒယ်ကို သုံးပြီး ဘယ်လောက်ထိ finetuning model က တိုးတက်မှုရှိသလဲ ဆိုတာကို လေ့လာကြည့်ကြရအောင်။  

ပြင်ထားတဲ့ shell script က အောက်ပါအတိုင်းပါ။  

In [172]:
!cat ./eval-optimize-10k.sh

#!/bin/bash
# Note the --stride 64. This gives the model context from previous sentences 
# when calculating PPL, leading to a much fairer (lower) score than stride=128.

time python transformer_lm.py --mode test \
                  --test_file ../data/10k_test.txt.clean \
                  --model_path ./myanmar_xglm_word_optimize \
                  --seq_len 128 \
                  --stride 64


In [173]:
!./eval-optimize-10k.sh

2026-05-02 00:09:48.310978: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 00:09:48.325226: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777655388.342309 3560950 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777655388.347894 3560950 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777655388.361877 3560950 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

ဒီတစ်ခါတော့ 1K test data နဲ့ evaluation လုပ်ကြည့်မယ်။  

In [170]:
!cat ./eval-optimize.sh

#!/bin/bash
# Note the --stride 64. This gives the model context from previous sentences 
# when calculating PPL, leading to a much fairer (lower) score than stride=128.

time python transformer_lm.py --mode test \
                  --test_file ../data/otest.word.clean \
                  --model_path ./myanmar_xglm_word_optimize \
                  --seq_len 128 \
                  --stride 64


In [171]:
!./eval-optimize.sh

2026-05-02 00:06:13.966489: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 00:06:13.979884: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777655173.995708 3560682 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777655174.000734 3560682 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777655174.013827 3560682 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

**Finetuning လုပ်တာက အော်ရဂျင်နယ်မော်ဒယ်ကို တိုက်ရိုက်သုံးတာထက် PPL ကျသွားတယ်။ ဆိုလိုတာက ရလဒ်ကောင်းတယ်ဆိုတာကိုတော့ သေချာပေါက် ပြောနိုင်ပါတယ်။** သို့သော်လည်း နှိုင်းယှဉ်ကြည့်ရင် တွေ့ရတဲ့အတိုင်းပဲ epoch 3 ခုနဲ့ ရလဒ်က epoch 10 မော်ဒယ်ထက် ပိုကောင်းနေတယ်ဆိုတာကို တွေ့ရတယ်။ လက်တွေ့မှာက PPL ရလဒ်ကိုပဲ ကြည့်ပြီးတော့လည်း ပြောရတာက ခက်ပါတယ်။ အထူးသဖြင့် အခုလိုမျိုး အရမ်း အပြတ်အသတ် ကွာမနေတဲ့ အခါမှာ။ ဘာပဲဖြစ်ဖြစ် လက်တွေ့အမျိုးမျိုး စမ်းကြည့်၊ အကဲဖြတ်ကြည့်မှပဲ အတိအကျ ပြောနိုင်ပါလိမ့်မယ်။ ဥပမာ PPL ရော text generation ရဲ့ quality ရော ကိုပါ နှိုင်းယှဉ်ကြည့်တာမျိုး။   

## Text Generation with Optimized XGLM Model

In [174]:
!cat ./gen-optimize.sh

#!/bin/bash
# Added --top_k and --top_p (Nucleus sampling) - standard practices for modern LLMs

python transformer_lm.py --mode generate \
                  --model_path ./myanmar_xglm_word_optimize \
                  --prompt "ပြည်ထောင်စု" \
                  --gen_length 200 \
                  --temperature 0.7 \
                  --top_k 50 \
                  --top_p 0.95


In [175]:
!./gen-optimize.sh

2026-05-02 00:19:48.887081: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 00:19:48.901696: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777655988.918871 3561361 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777655988.924405 3561361 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777655988.938847 3561361 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [176]:
!./gen-optimize.sh

2026-05-02 00:20:28.675238: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 00:20:28.689541: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777656028.706473 3561450 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777656028.711986 3561450 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777656028.725956 3561450 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [177]:
!./gen-optimize.sh

2026-05-02 00:21:08.868092: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 00:21:08.881482: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777656068.897375 3561553 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777656068.902452 3561553 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777656068.915788 3561553 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Model Folder Structure  

မော်ဒယ်တွေကို သိမ်းထားတဲ့ ဖိုလ်ဒါကိုလည်း လေ့လာနိုင်အောင် ဒီ Notebook မှာ ရိုက်ပြပေးထားလိုက်ပါမယ်။  

In [179]:
!tree ./myanmar_xglm_word/

./myanmar_xglm_word/
├── checkpoint-1044
│   ├── config.json
│   ├── generation_config.json
│   ├── model.safetensors
│   ├── optimizer.pt
│   ├── rng_state.pth
│   ├── scheduler.pt
│   ├── sentencepiece.bpe.model
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   ├── trainer_state.json
│   └── training_args.bin
├── checkpoint-1566
│   ├── config.json
│   ├── generation_config.json
│   ├── model.safetensors
│   ├── optimizer.pt
│   ├── rng_state.pth
│   ├── scheduler.pt
│   ├── sentencepiece.bpe.model
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   ├── trainer_state.json
│   └── training_args.bin
├── checkpoint-522
│   ├── config.json
│   ├── generation_config.json
│   ├── model.safetensors
│   ├── optimizer.pt
│   ├── rng_state.pth
│   ├── scheduler.pt
│   ├── sentencepiece.bpe.model
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   ├── trainer_state.json
│   └── train

In [180]:
!tree ./myanmar_xglm_word_optimize/

./myanmar_xglm_word_optimize/
├── checkpoint-1036
│   ├── config.json
│   ├── generation_config.json
│   ├── model.safetensors
│   ├── optimizer.pt
│   ├── rng_state.pth
│   ├── scheduler.pt
│   ├── sentencepiece.bpe.model
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   ├── trainer_state.json
│   └── training_args.bin
├── checkpoint-1295
│   ├── config.json
│   ├── generation_config.json
│   ├── model.safetensors
│   ├── optimizer.pt
│   ├── rng_state.pth
│   ├── scheduler.pt
│   ├── sentencepiece.bpe.model
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   ├── trainer_state.json
│   └── training_args.bin
├── checkpoint-1554
│   ├── config.json
│   ├── generation_config.json
│   ├── model.safetensors
│   ├── optimizer.pt
│   ├── rng_state.pth
│   ├── scheduler.pt
│   ├── sentencepiece.bpe.model
│   ├── special_tokens_map.json
│   ├── tokenizer_config.json
│   ├── tokenizer.json
│   ├── trainer_state.json
│  

## RAG with a Modern 8B-Class Model (Quantized on GPU)  

24GB VRAM means we don't have to settle for GPT-2 for RAG. We can load Gemma-2-2b-it (or even Llama-3-8B) in 4-bit quantization. It uses ~5GB VRAM, leaving plenty of room. We use sentence-transformers on GPU for blazing-fast retrieval.  

ပထမတော့ အထက်မှာ ရေးထားတဲ့အတိုင်းပဲ quantization လုပ်ဖို့ ကြိုးစားသေးတယ်။  float32 ကို bfloat16 ပြောင်းသုံးဖို့ကိစ္စပါ။  

#!pip install -q bitsandbytes --break-system-packages
 
တစ်ခုရှိတာက ဒီ bitsandbytes library က complex C++/CUDA library ဖြစ်တာကြောင့် ဆရာ့စက်မှာ အဆင်မပြေဘူး။ ဆရာ့စက်မှာ VRAM က Gemma-2-2b-it ကို load လုပ်လို့ ရပါတယ်။ အဲဒါကြောင့် bitsandbytes ကို မသုံးပဲသွားမယ်။  

ကျောင်းသားတွေက ကိုယ့်စက်ရဲ့ VRAM ပေါ်မူတည်ပြီး လိုအပ်ရင် bitsandbytes library ကို သုံးပြီး quantization လုပ်ပါ။   

ပြီးတော့ sentence-transformers library က ကိုယ့်စက်ထဲမှာ မရှိရင်လည်း installation လုပ်ရလိမ့်မယ်။  

## Why RAG?  

**The Problem:** LLMs တွေက အတိအကျမှတ်ထားတာမျိုး မဟုတ်ဘူး။ တနည်းအားဖြင့်က မေ့တတ်တယ်။  
အဲဒါကြောင့် ငါတို့က LLM တစ်ခု (Gemma) ကို ငါတို့ မေလ ၁ရက်နေ့က trained ခဲ့တာကို မှတ်မိသေးလား။ အဲဒီတုန်းက LSTM ရဲ့ PPL က ဘယ်လောက်ရခဲ့တာလဲ လို့ မေးရင် သူမဖြေပေးနိုင်ပါဘူး။ ဘာဖြစ်လို့လဲ ဆိုတော့ LLM မော်ဒယ်တွေက အင်တာနက်က ရှိသမျှ စာသားတွေ (i.e. terabytes of internet text) ကို အကုန် ချုံ့သိမ်းထားတာမျိုးပါ။ လက်တွေ့မှာက weight တန်ဖိုးတွေအနေနဲ့ပဲ သူသိတာပါ။ သူက တိတိကျကျ မှတ်သားထားတဲ့ memory လိုမျိုး မရှိပါဘူး။ သူက pattern တွေအနေနဲ့ပဲ လေ့လာထားပြီး အဲဒီကနေမှာ ဖြစ်နိုင်တဲ့ စာသားတွေကို ခန့်မှန်းပြီး ကျပန်း ထုတ်ပေးနေတာပါ။ အဲဒါကြောင့် သူ့ရဲ့ အဖြေတွေက ရွှီးတာမျိုး၊ မှန်သလိုလို ဖြစ်နေတာမျိုးနဲ့ မှားနေတာတွေ အများကြီး ဖြစ်နိုင်ပါတယ်။  

**The RAG Solution:** အဲဒီ ပြဿနာကို သုတေသနပညာရှင်တွေက RAG ဆိုတဲ့ နည်းလမ်းနဲ့ ပြေလည်နိုင်မလား ဆိုပြီး ကြိုးစားကြည့်နေကြတာပါ။ ကိုယ့် ဒိုမိန်းနဲ့ ဆိုင်တဲ့ memory ကို ထပ်ဖြည့်တဲ့ ပုံစံမျိုးပါပဲ။ RAG ရဲ့ အရှည်က Retrieval-Augmented Generation လို့ခေါ်ဆိုပြီးတော့ AI ကို Retriever နဲ့  Generator ဆိုပြီး နှစ်ပိုင်းခွဲလိုက်တဲ့ ပုံစံမျိုးပါ။  

**The Retriever (SentenceTransformer):** ဒါက ကိုယ့်ဒိုမိန်းနဲ့ ဆိုင်တဲ့ အချက်အလက် အတိအကျမှတ်သားထားတဲ့ မှတ်ဉာဏ် (i.e. memory) ပါပဲ။ LLM ကို မရမက သင်ပေး၊ မှတ်ခိုင်းနေတာထက် ကိုယ်ကသုံးမယ့် ဒေတာဗေစ့် ကို vector database အနေနဲ့ ကြိုတင် ပြင်ဆင်ထားတဲ့ သဘောပါ။ Python code ထဲမှာ သုံးထားတဲ့ kb_embs လိုမျိုးပါ။ ယူဇာက မေးခွန်းမေးလိုက်တိုင်းမှာ Retriever က အဲဒီကြိုပြင်ထားတဲ့ vector database ထဲကနေ Dot Product/Cosine Similarity တို့နဲ့ ရှာဖွေပြီး အဖြစ်နိုင်ဆုံး (သို့) သက်ဆိုင်မှုအများဆုံး စာပိုဒ်တွေကို ဆွဲထုတ် ယူလာပါတယ်။ 

**The Generator (Gemma-2):** Generator ဆိုတာကတော့ "Reasoning Engine" ပါပဲ။ အကျိုးသင့်၊ အကြောင်းသင့် logically ကျ၊ မကျ ကို ခန့်မှန်းပေးတဲ့ အင်ဂျင်ပါပဲ။ Retriever ကဆွဲထုတ်ပေးတဲ့ စာပိုဒ်တွေကို text prompt ထဲမှာပါ ဖြည့်စွက်ပြီးမှ၊  ခုပြင်ပေးထားတဲ့ စာပိုဒ်တွေကိုဖတ်ကြည့်ပြီး အဖြေပေးပါ ဆိုတဲ့ ပုံစံမျိုးနဲ့ LLM ကို ခိုင်းတဲ့ သဘောပါပဲ။   

RAG ကို သုံးခြင်းအားဖြင့် အားသာချက်ကတော့ text ဖိုင် သို့မဟုတ် databaseကို ပြင်တာ၊ ဖြည့်စွက်တာ လုပ်ပြီး knowledge ကို တိုးသွားလို့ ရပါတယ်။ LLM ကို ပြန် train လုပ်တာမျိုး၊ finetuning လုပ်တာမျိုး လုပ်ဖို့ မလိုတဲ့အတွက် ကုန်ကျစရိတ် သက်သာစေတဲ့ အပိုင်းပါပဲ။ အဲဒါကြောင့် ကုမ္ပဏီတွေမှာ RAG ကို အသုံးချနေကြတာတွေ အများကြီးရှိပါတယ်။    

In [94]:
from sentence_transformers import SentenceTransformer
import numpy as np
import textwrap
import torch

# 1. GPU-Accelerated Embeddings
print("Loading Retriever...")
encoder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# Our tiny mock Knowledge Base (Vector DB)
kb = [
    "Hugging Face Transformers is a library for state-of-the-art NLP.",
    "Retrieval-Augmented Generation helps reduce hallucinations by grounding LLMs on retrieved documents.",
    "Perplexity is the exponentiated average negative log-likelihood.",
    "Cross-entropy between true distribution p and model q is H(p,q) = -Σ p(x) log q(x).",
]
kb_embs = encoder.encode(kb, convert_to_numpy=True, normalize_embeddings=True)

def retrieve_top_k(query: str, k=2):
    q_emb = encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    # Dot product similarity (because vectors are L2-normalized, this equals Cosine Similarity)
    scores = kb_embs @ q_emb
    top_idx = np.argsort(scores)[::-1][:k]
    return [(kb[i], float(scores[i])) for i in top_idx]

# 2. Load Generator (Natively in bfloat16 - No bitsandbytes needed!)
print("Loading Generator (bfloat16)...")
from transformers import AutoTokenizer, AutoModelForCausalLM

# Using a modern instruction-tuned model
gen_model_id = "google/gemma-2-2b-it" 
tokenizer_gen = AutoTokenizer.from_pretrained(gen_model_id)
# We just use torch_dtype=torch.bfloat16. Simple, fast, stable.
model_gen = AutoModelForCausalLM.from_pretrained(
    gen_model_id, 
    torch_dtype=torch.bfloat16,
    device_map="auto" # Automatically puts it on your RTX 3090
).to(device)

# 3. RAG Pipeline
def rag_answer(query: str):
    # Step A: Retrieve relevant documents based on semantic similarity
    hits = retrieve_top_k(query, k=2)
    context = "\n".join([f"- {txt}" for txt, _ in hits])
    
    # Step B: Construct a prompt forcing the LLM to use ONLY the context
    prompt = textwrap.dedent(f"""\
    Answer the question based only on the provided context.
    
    Context:
    {context}
    
    Question: {query}
    
    Answer:""").strip()
    
    inputs = tokenizer_gen(prompt, return_tensors="pt").to(device)
    
    # Step C: Generate the answer
    with torch.no_grad():
        outputs = model_gen.generate(**inputs, max_new_tokens=100, temperature=0.1)
        
    answer = tokenizer_gen.decode(outputs[0], skip_special_tokens=True)
    return answer.split("Answer:")[-1].strip()

print("\n--- RAG Inference ---")
print("Q: How is RAG useful?")
print("A:", rag_answer("How is RAG useful?"))
print("\nQ: What is the math behind Perplexity?")
print("A:", rag_answer("What is the math behind Perplexity?"))

Loading Retriever...


/home/ye/.local/lib/python3.12/site-packages/torch/nn/modules/module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Loading Generator (bfloat16)...


Loading checkpoint shards: 100%|███████████████████████████████████████████| 2/2 [00:01<00:00,  1.09it/s]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- RAG Inference ---
Q: How is RAG useful?


/home/ye/.local/lib/python3.12/site-packages/torch/nn/modules/module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


A: RAG helps reduce hallucinations by grounding LLMs on retrieved documents.

Q: What is the math behind Perplexity?
A: Perplexity is the exponentiated average negative log-likelihood.


## RAG for Myanmar Language 

ဒီတခါတော့ မြန်မာစာနဲ့ RAG ကို ဆောက်ပြပြီး မေးခွန်းမေးကြည့်ကြရအောင်။  

BBC article link: [https://www.bbc.com/burmese/articles/cjr9r8veve0o](https://www.bbc.com/burmese/articles/cjr9r8veve0o)  


In [1]:
%cd /mnt/disk1/ye/exp/LM-Tutorial/transformer

/mnt/disk1/ye/exp/LM-Tutorial/transformer


In [11]:
import textwrap
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# ==========================================
# 1. THE KNOWLEDGE BASE (The BBC Article)
# ==========================================
# In a real app, this would be thousands of documents chunked and stored in a Vector DB.
bbc_paragraphs = [
    "မြန်မာကျပ် မလို တရုတ် ယွမ် သာ လိုတဲ့ ရှမ်းပြည်က နယ်စပ်မြို့တွေ တရုတ်-မြန်မာ နယ်စပ် မြို့တွေ မှာ မြန်မာ ကျပ်ငွေ ထက် တရုတ် ငွေကြေး ယွမ်ကို ဦးစားပေး သုံးတာ က အရင်ကတည်းက ရှိခဲ့ပြီး မဆန်းလှပါဘူး။",
    "၂၀၂၇ စစ်ဆင်ရေးအလွန်မှာတော့ ရှမ်းပြည်နယ်(မြောက်ပိုင်း)မှာ ယွမ်တန်ဖိုးက ဘယ်တုန်းကမှ နဲ့မတူအောင် မြင့်တက် လာပြီး ကျပ်ငွေ က နောက်ကောက်ကျန်ရစ်ခဲ့ ပါတယ်။ လိုချင်သူ ရှားလာ ပါတယ်။",
    "ဝ တပ်ဖွဲ့ (UWSA)နဲ့ ကိုးကန့်တပ်(MNDAA) ထိန်းချုပ်ထားတဲ့ မြို့တွေမှာ ကျပ်ငွေအစား ယွမ်ငွေကိုပဲ ဘာလို့ သုံးလာကြတာလဲ။ ရှမ်းမြောက်က သိန္နီ၊ ကွမ်လုံ၊ လောက်ကိုင်နဲ့ ချင်းရွှေဟော်မြို့တွေမှာ အရင်ကထက် တရုတ်ယွမ်ငွေသုံးစွဲတာ ပိုပြီးတွင်ကျယ်လာတယ်လို့ အဲဒီနေရာတွေမှာ နေထိုင်နေကြတဲ့ ဒေသခံတချို့က ဘီဘီစီကိုပြောပါတယ်။",
    "အဲဒီမြို့တွေကို ၂၀၂၃ ခုနှစ်၊ အောက်တိုဘာ ၂၇ ရက်က စတင်ခဲ့တဲ့ မြောက်ပိုင်းညီနောင် သုံးဖွဲ့ရဲ့ ၁၀၂၇ စစ်ဆင်ရေး ပထမပိုင်းမှာပဲ ကိုးကန့်တပ်က ထိန်းချုပ်လုက်တာဖြစ်ပါတယ်။",
    "သိန္နီဒေသခံတစ်ဦးကတော့ မြို့ထဲက ကုန်ခြောက်ဆိုင်၊ သစ်သီးဆိုင်၊ စားသောက်ဆိုင်နဲ့ ဈေးတွေမှာ ကျပ်ငွေနဲ့ ပေးချေလို့ရပေမဲ့ ကျပ်ငွေသုံးမယ်ဆိုရင် ကုိင်ရတဲ့ ပိုက်ဆံအရွက်ရေက ပိုများတာကြောင့် ယွမ်ငွေကို ပိုပြီး သုံးနေရတယ်လို့ ဆိုပါတယ်။"
]

# ==========================================
# 2. THE MULTILINGUAL RETRIEVER
# ==========================================
print("Loading Multilingual Retriever...")
# We MUST use a multilingual model here. English-only MiniLM will fail on Myanmar.
encoder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)

print("Embedding Knowledge Base...")
kb_embs = encoder.encode(bbc_paragraphs, convert_to_numpy=True, normalize_embeddings=True)

def retrieve_top_k(query: str, k=2):
    q_emb = encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    scores = kb_embs @ q_emb
    top_idx = np.argsort(scores)[::-1][:k]
    return [bbc_paragraphs[i] for i in top_idx]

# ==========================================
# 3. THE GENERATOR (Gemma 2)
# ==========================================
#print("Loading Generator...")
#gen_model_id = "google/gemma-2-2b-it" 
#tokenizer_gen = AutoTokenizer.from_pretrained(gen_model_id)
#model_gen = AutoModelForCausalLM.from_pretrained(
#    gen_model_id, 
#    torch_dtype=torch.bfloat16,
#    device_map="auto"
#).to(device)

# ==========================================
# 3. THE GENERATOR (Gemma 2)
# ==========================================
print("Loading Generator...")
gen_model_id = "google/gemma-2-2b-it" 
tokenizer_gen = AutoTokenizer.from_pretrained(gen_model_id)

# THE SILVER BULLET FIX:
# 1. device_map="auto" safely handles VRAM and prevents OOMs.
# 2. attn_implementation="eager" disables PyTorch's torch.compile, 
#    permanently preventing the FailOnRecompileLimitHit error in Jupyter!
model_gen = AutoModelForCausalLM.from_pretrained(
    gen_model_id, 
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager" 
)

# ==========================================
# 4. RAG PIPELINE (Myanmar Prompt)
# ==========================================
def rag_answer_myanmar(query: str):
    # Step A: Retrieve context in Myanmar
    retrieved_docs = retrieve_top_k(query, k=2)
    context_string = "\n".join([f"- {doc}" for doc in retrieved_docs])
    
    # Step B: Create a Myanmar prompt instructing the LLM to read and extract
    # We explicitly tell it to rely ONLY on the text to prevent hallucinations
    prompt = f"""အောက်ပါ စာသားအချက်အလက်များကို ဖော်ပြထားပါသည်။ ထိုစာသားများအပေါ် အခြေခံပြီး မေးခွန်းကို မြန်မာလို ဖြေပေးပါ။ 

စာသားအချက်အလက်များ:
{context_string}

မေးခွန်း: {query}

အဖြေ: """
    
    inputs = tokenizer_gen(prompt, return_tensors="pt").to(device)
    
    # Step C: Generate
    with torch.no_grad():
        outputs = model_gen.generate(
            **inputs, 
            max_new_tokens=150, 
            pad_token_id=tokenizer_gen.eos_token_id,
            do_sample=False # Greedy decoding for factual extraction
        )
        
    # Extract just the generated part
    answer = tokenizer_gen.decode(outputs[0], skip_special_tokens=True)
    return answer.split("အဖြေ:")[-1].strip()

# ==========================================
# 5. RUN THE DEMO
# ==========================================
print("\n" + "="*50)
print("--- MYANMAR RAG INFERENCE ---")
print("="*50)

q1 = "ဘာကြောင့် ယွမ်ငွေ သုံးသလဲ"
print(f"\nမေးခွန်း ၁: {q1}")
print(f"အဖြေ: {rag_answer_myanmar(q1)}\n")

q2 = "ရှမ်းမြောက် ဘယ်မြို့တွေ မှာလဲ"
print(f"\nမေးခွန်း ၂: {q2}")
print(f"အဖြေ: {rag_answer_myanmar(q2)}")

Loading Multilingual Retriever...
Embedding Knowledge Base...
Loading Generator...


Loading checkpoint shards: 100%|███████████████████████████████████████████| 2/2 [00:01<00:00,  1.04it/s]



--- MYANMAR RAG INFERENCE ---

မေးခွန်း ၁: ဘာကြောင့် ယွမ်ငွေ သုံးသလဲ
အဖြေ: ကျပ်ငွေ သုံးမယ်ဆိုရင် ကုိင်ရတဲ့ ပိုက်ဆံအရွက်ရေက ပိုများတာကြောင့် ယွမ်ငွေကို ပိုပြီး သုံးနေရတယ်လို့ ဆိုပါတယ်။ 

ကျပ်ငွေ သုံးမယ်ဆိုရင် ကုိင်ရတဲ့ ပိုက်ဆံအရွက်ရေက ပိုများတာကြောင့် ယ


မေးခွန်း ၂: ရှမ်းမြောက် ဘယ်မြို့တွေ မှာလဲ


/home/ye/.local/lib/python3.12/site-packages/torch/nn/modules/module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


အဖြေ: **ရှမ်းမြောက် ဘယ်မြို့တွေ မှာလဲ**

**အဲဒီမြို့တွေကို ၂၀၂၃ ခုနှစ်၊ အောက်တိုဘာ ၂၇ ရက်က စတင်ခဲ့တဲ့ မြောက်ပိုင်းညီနောင် သုံးဖွဲ့ရဲ့ ၁၀၂၇ စစ်ဆင်ရေး ပထမပိုင်းမှာပဲ ကိုးကန့်တပ်က ထိန်းချုပ်လုက်တာဖြ


# Q&A RAG

ဒီတစ်ခါတော့ paragraph ပုံစံမျိုးနဲ့ RAG ကို ဆောက်တာမဟုတ်ပဲ။ အမေး၊ အဖြေ ပုံစံမျိုးနဲ့ ဆောက်ကြည့်ကြမယ်။  

In [10]:
import numpy as np

# ==========================================
# 1. STRUCTURED QA KNOWLEDGE BASE
# ==========================================
# Storing as a list of dictionaries allows us to separate the Question from the Answer
qa_database = [
    {"q": "မေလ ၂ ရက်နေ့ အတန်းမှာ ဘာ သင် မှာလဲ", "a": "မြန်မာစာ နဲ့ ထိုင်းစာ ကို ကွန်ပျူတာ သုံးပြီး ဘာသာပြန် မယ်"},
    {"q": "မေလ ၁ ရက်နေ့ က ထိုင်းမှာ ရုံးပိတ်လား", "a": "ပိတ်တယ်"},
    {"q": "ဆရာ က စက်ဝိုင်း နဲ့ တြိဂံ ဘယ်ဟာ ကို ပိုကြိုက် သလဲ", "a": "စက်ဝိုင်းကို ပို ကြိုက်တယ်"},
    {"q": "ဆရာ က ဘယ်လမှာ မွေးတာလဲ", "a": "ဧပြီ လ မှာ"},
    {"q": "မလှ က ကျောင်းပြေးဖူးလား", "a": "သုံးခါပဲ ပြေးဖူးပါတယ်"}
]

# ==========================================
# 2. EMBEDDING THE QA PAIRS
# ==========================================
print("Embedding QA Database...")
# We embed "Q: ... A: ..." together so the Retriever learns the mapping between them
qa_texts = [f"Q: {item['q']} A: {item['a']}" for item in qa_database]
qa_embs = encoder.encode(qa_texts, convert_to_numpy=True, normalize_embeddings=True)

def retrieve_exact_qa(query: str):
    q_emb = encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    scores = qa_embs @ q_emb
    
    # Get the index of the highest scoring match
    best_idx = np.argmax(scores)
    best_score = scores[best_idx]
    
    # Return the dictionary and the similarity score (useful for debugging)
    return qa_database[best_idx], float(best_score)

# ==========================================
# 3. STRICT RAG PIPELINE FOR EXTRACTING ANSWERS
# ==========================================
def rag_exact_answer(query: str):
    # Step A: Retrieve the best matching QA pair
    best_match, score = retrieve_exact_qa(query)
    
    # Step B: Create a strict prompt
    context = f"Q: {best_match['q']}\nA: {best_match['a']}"
    
    prompt = f"""ဒါက သိထားရမည့် မေးခွန်းနဲ့ အဖြေ ဖြစ်ပါတယ်။ 
မေးခွန်းက အောက်ပါ မေးခွန်းနဲ့ ကိုက်ညီလျှင် အဖြေကို အဲဒီအတိုင်း ပြန်လည်ထုတ်ပေးပါ။ 
မမှန်ကန်လျှင် "မမှန်ပါ" လို့သာ ပြောပေးပါ။

စာသားအချက်အလက်:
{context}

မေးခွန်း: {query}

အဖြေ:"""

    inputs = tokenizer_gen(prompt, return_tensors="pt").to(device)
    
    # Step C: Generate
    with torch.no_grad():
        outputs = model_gen.generate(
            **inputs, 
            max_new_tokens=50,  # Answers are short, we don't need 150 tokens
            pad_token_id=tokenizer_gen.eos_token_id,
            do_sample=False 
        )
        
    answer = tokenizer_gen.decode(outputs[0], skip_special_tokens=True)
    final_answer = answer.split("အဖြေ:")[-1].strip()
    
    return final_answer, score

# ==========================================
# 4. RUN THE DEMO
# ==========================================
print("\n" + "="*50)
print("--- EXACT QA RAG INFERENCE ---")
print("="*50)

# Test 1: Exact Match
q1 = "ဆရာ က ဘယ်လမှာ မွေးတာလဲ"
print(f"\nမေးခွန်း ၁ (အတိအကျကိုက်ညီ): {q1}")
ans1, score1 = rag_exact_answer(q1)
print(f"Retriever Score: {score1:.2f} (1.0 is a perfect match)")
print(f"အဖြေ: {ans1}")

# Test 2: Paraphrased Match (Testing the Retriever's intelligence)
q2 = "စတုဂံ ကို ပိုကြိုက်တယ်" 
print(f"\nမေးခွန်း ၂ (အဓိပ္ပာယ် နားလည် မလည်စမ်းတာ): {q2}")
ans2, score2 = rag_exact_answer(q2)
print(f"Retriever Score: {score2:.2f} (Lower score)")
print(f"အဖြေ: {ans2}")

# Test 3: Out of Domain (Testing the strict prompt)
q3 = "ဘာကြောင့် ယွမ် သုံးတာလဲ"
print(f"\nမေးခွန်း ၃ (ဒိုမိန်း မတူတာကိုစမ်းတာ): {q3}")
ans3, score3 = rag_exact_answer(q3)
print(f"Retriever Score: {score3:.2f} (Very low score)")
print(f"အဖြေ: {ans3}")

Embedding QA Database...

--- EXACT QA RAG INFERENCE ---

မေးခွန်း ၁ (အတိအကျကိုက်ညီ): ဆရာ က ဘယ်လမှာ မွေးတာလဲ
Retriever Score: 0.87 (1.0 is a perfect match)
အဖြေ: ဆရာ က ဘယ်လမှာ မွေးတာလဲ ဧပြီ လ မှာ 

**ကျွန်တော် ကျွန်တော်

မေးခွန်း ၂ (အဓိပ္ပာယ် နားလည် မလည်စမ်းတာ): စတုဂံ ကို ပိုကြိုက်တယ်
Retriever Score: 0.47 (Lower score)
အဖြေ: စက်ဝိုင်း ကို ပိုကြိုက်တယ် 

**ကျွန်တော် ကျွန်တော် ကျွန်တော် ကျွန်တော

မေးခွန်း ၃ (ဒိုမိန်း မတူတာကိုစမ်းတာ): ဘာကြောင့် ယွမ် သုံးတာလဲ
Retriever Score: 0.29 (Very low score)
အဖြေ: ဘာကြောင့် ယွမ် သုံးတာလဲ 

**ကျွန်တော် ကျွန်တော် ကျွန်တော် က


## The Evaluation

We cannot directly compare the PPL of KenLM, LSTM, and Transformers.

- KenLM calculates PPL per Word (e.g., predicting 1 out of 10,000 words).
- XGLM calculates PPL per Subword (e.g., breaking "မြန်မာ" into 3 pieces, predicting 1 out of 256,000 pieces).
- Because XGLM makes predictions more frequently (due to smaller pieces), its PPL mathematically looks much higher/worse, even if it's a better model!

**The Solution:** Bits-Per-Character (BPC).
BPC normalizes everything back to raw characters. It asks: "How many bits of information does the model need to guess the next single character?"

**KenLM BPC:** ```(log10 prob of a word * log2(10)) / number of characters in that word```  
By comparing BPC, we are finally comparing Apples to Apples.

ရှေ့မှာက KenLM ကို toy ဒေတာနဲ့ပဲ ဆောက်ရသေးတာမို့ KenLM ကိုလည်း myPOS corpus နဲ့ အရင်ဆောက်ပါမယ်။ ပြီးမှ LSTM, Transformer တို့နဲ့အတူ နှိုင်းယှဉ်ဖို့ လုပ်ပါမယ်။ မဟုတ်ရင် KenLM နဲ့ ဆောက်ထားတဲ့ corpus က တအားသေးနေတာမို့ နှိုင်းယှဉ်လို့ အဆင်မပြေပါ။  

In [96]:
%cd /mnt/disk1/ye/exp/LM-Tutorial

/mnt/disk1/ye/exp/LM-Tutorial


In [20]:
!mkdir kenlm

In [22]:
# -o 5: 5-gram model
# -S 16G: Use 16GB of RAM (speeds up disk sorting)
# -T: Use your fast NVMe drive for temp files
!lmplz -o 5 -S 16G -T /tmp/kenlm_tmp < ./data/mypos_v3.word.clean > kenlm/myanmar_lm.arpa

=== 1/5 Counting and sorting n-grams ===
Reading /mnt/disk1/ye/exp/LM-Tutorial/data/mypos_v3.word.clean
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 510463 types 24719
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:296628 2:1676055808 3:3142604800 4:5028167168 5:7332744192
Statistics:
1 24719 D1=0.679643 D2=1.0199 D3+=1.3394
2 173900 D1=0.763124 D2=1.12194 D3+=1.42456
3 335010 D1=0.867187 D2=1.26549 D3+=1.41956
4 397924 D1=0.932677 D2=1.34069 D3+=1.50121
5 398900 D1=0.948056 D2=1.45671 D3+=1.63352
Memory estimate for binary LM:
type       kB
probing 28893 assuming -p 1.5
probing 34303 assuming -r models -p 1.5
trie    13557 without quantization
trie     7241 assuming -q 8 -b 8 quantization 
trie    12256 assuming -a 22 array pointer compression
trie     5940 assuming -a 22 -q 8 -b 8 array pointer 

## Convert ARPA to Binary

Binary ဖိုင်အဖြစ်ပြောင်းထားလိုက်ရင် Jupyter Notebook ထဲကနေ ခေါ်သုံးရင်လည်း မြန်ပါတယ်။  

In [23]:
!build_binary ./kenlm/myanmar_lm.arpa ./kenlm/myanmar_lm.binary

Reading ./kenlm/myanmar_lm.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
SUCCESS


In [24]:
!ls -lh ./kenlm/*

-rw-rw-r-- 1 ye ye 95M May  2 03:20 ./kenlm/myanmar_lm.arpa
-rw-r--r-- 1 ye ye 29M May  2 03:22 ./kenlm/myanmar_lm.binary


## Prepare KenLM Evaluation Python Code

KenLM မှာက PPL ကို တွက်ဖို့ commandline tool အနေနဲ့ ရေးမထားပါဘူး။ အဲဒါကြောင့် python code တစ်ပုဒ်ရေးလိုက်ပါတယ်။ အောက်ပါအတိုင်းပါ။  

In [20]:
!cat ./kenlm/eval_kenlm.py

import kenlm, sys, math

model_path = sys.argv[1]
test_path = sys.argv[2]

model = kenlm.LanguageModel(model_path)
sum_log10 = 0.0
n_words = 0

with open(test_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        for prob, length, oov in model.full_scores(line):
            sum_log10 += prob
        n_words += len(line.split())

# Convert log10 to natural log, then calculate PPL
#sum_nats = sum_log10 * math.log(10)
#ppl = math.exp(-sum_nats / n_words) if n_words else float("inf")
#print(f"PPL: {ppl:.2f}")

# Cleaner, unambiguous version
sum_nats = -sum_log10 * math.log(10)
ppl = math.exp(sum_nats / n_words)
entropy = sum_nats / n_words
print(f"PPL: {ppl:.2f}")
print(f"Entropy (nats): {entropy:.4f}")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [102]:
!python ./kenlm/eval_kenlm.py ./kenlm/myanmar_lm.binary ./data/otest.word.clean

PPL: 9.10
Entropy (nats): 2.2085


In [10]:
%cd /mnt/disk1/ye/exp/LM-Tutorial/

/mnt/disk1/ye/exp/LM-Tutorial


In [19]:
import math
import json
import pandas as pd
import torch
from pathlib import Path
import sys

sys.path.append(str(Path.cwd() / "lstm"))
sys.path.append(str(Path.cwd() / "transformer"))

import kenlm
from lstm_lm import TextDataset as LSTM_Dataset, LSTM_LM as LSTM_Model, Vocabulary
from transformer_lm import evaluate_ppl as transformer_evaluate_ppl

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

TEST_FILE = "./data/otest.word.clean"

# ==========================================
# 1. EVALUATE KENLM (N-gram)
# ==========================================
print("Evaluating KenLM...")
# CHANGE THIS to your new binary file!
model_kenlm = kenlm.LanguageModel("./kenlm/myanmar_lm.binary") 

def get_kenlm_metrics(path):
    sum_log10, n_words, n_chars = 0.0, 0, 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            for prob, length, oov in model_kenlm.full_scores(line):
                sum_log10 += prob
                n_words += 1           # counts </s> token too
            n_chars += len(line)

    # sum_log10 is negative → negate once for all positive quantities
    sum_nats   = -sum_log10 * math.log(10)
    total_bits = -sum_log10 * math.log2(10)

    ppl_kenlm = math.exp(sum_nats / n_words)  if n_words else float("inf")
    ent_kenlm = sum_nats / n_words             if n_words else float("inf")
    bpc_kenlm = total_bits / n_chars           if n_chars else float("inf")

    return ppl_kenlm, ent_kenlm, bpc_kenlm

ppl_kenlm, ent_kenlm, bpc_kenlm = get_kenlm_metrics(TEST_FILE)

# ==========================================
# 2. EVALUATE LSTM (Word-level)
# ==========================================
print("Evaluating LSTM...")
with open("./lstm/myanmar_word_lm.pt.vocab", "r", encoding="utf-8") as f:
    token2idx = json.load(f)
vocab_lstm = Vocabulary()
vocab_lstm.token2idx = token2idx
vocab_lstm.idx2token = {int(v): k for k, v in token2idx.items()}

model_lstm = LSTM_Model(len(vocab_lstm)).to(device)
model_lstm.load_state_dict(torch.load("./lstm/myanmar_word_lm.pt", map_location=device))
model_lstm.eval()

with open(TEST_FILE, 'r', encoding='utf-8') as f:
    test_tokens = f.read().split()

# The production LSTM uses sequences of length 50
test_ds = LSTM_Dataset(test_tokens, vocab_lstm, seq_len=50)

# Use the exact same loss function used during training
criterion = torch.nn.CrossEntropyLoss()
nll_sum, n_tokens = 0.0, 0

with torch.no_grad():
    for i in range(len(test_ds)):
        x, y = test_ds[i]
        
        # Add batch dimension: shape (50,) becomes (1, 50)
        x = x.unsqueeze(0).to(device)
        y = y.unsqueeze(0).to(device)
        
        # Forward pass
        out, _ = model_lstm(x)
        
        # Calculate loss (identical to training loop in lstm_lm.py)
        # out shape: (1, 50, vocab_size)
        # y shape: (1, 50)
        loss = criterion(out.view(-1, out.size(-1)), y.view(-1))
        
        # Accumulate total negative log likelihood
        nll_sum += loss.item() * y.numel()
        n_tokens += y.numel()

# Calculate final metrics
ppl_lstm = math.exp(nll_sum / n_tokens)
ent_lstm = nll_sum / n_tokens

# ==========================================
# 3. EVALUATE TRANSFORMER (XGLM)
# ==========================================
print("Evaluating XGLM...")
class Args:
    pass
args = Args()
args.seq_len = 128
args.stride = 64

from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer_xglm = AutoTokenizer.from_pretrained("./transformer/myanmar_xglm_word")
model_xglm = AutoModelForCausalLM.from_pretrained(
    "./transformer/myanmar_xglm_word", 
    torch_dtype=torch.bfloat16, 
    device_map="auto", 
    attn_implementation="eager"
)

with open(TEST_FILE, 'r', encoding='utf-8') as f:
    test_lines_xglm = [line.strip() for line in f if line.strip()]

ppl_xglm, nll_xglm = transformer_evaluate_ppl(args, model_xglm, tokenizer_xglm, test_lines_xglm)
ent_xglm = nll_xglm 

# ==========================================
# 4. COMPILE THE RESULTS TABLE
# ==========================================
results = [
    {
        "Model": "KenLM (Myanmar 5-gram)", 
        "Test PPL": f"{ppl_kenlm:.2f}", 
        "Test Entropy (nats)": f"{ent_kenlm:.2f}",
        "Test BPC": f"{bpc_kenlm:.3f}" 
    },
    {
        "Model": "PyTorch LSTM (GPU)", 
        "Test PPL": f"{ppl_lstm:.2f}", 
        "Test Entropy (nats)": f"{ent_lstm:.2f}",
        "Test BPC": "N/A (Word level)" 
    },
    {
        "Model": "XGLM (Transformer, GPU)", 
        "Test PPL": f"{ppl_xglm:.2f}", 
        "Test Entropy (nats)": f"{ent_xglm:.2f}",
        "Test BPC": "N/A (Subword level)"
    }
]

df = pd.DataFrame(results)
print(df)

Using device: cuda
Evaluating KenLM...
Evaluating LSTM...
Evaluating XGLM...
Tokenizing test set (1000 lines)...
Evaluating with Stride=64, Context=128...
                     Model Test PPL Test Entropy (nats)             Test BPC
0   KenLM (Myanmar 5-gram)     7.70                2.04                0.586
1       PyTorch LSTM (GPU)     2.12                0.75     N/A (Word level)
2  XGLM (Transformer, GPU)    18.21                2.90  N/A (Subword level)


## SRILM with Myanmar Corpus

စာသင်ပြီးသွားတော့ SRILM ကိုလည်း အထက်က ဇယားမှာထည့်ပြီး နှိုင်းယှဉ်ချင်တာနဲ့ SRILM ကိုလည်း myPOS corpus နဲ့ ဆောက်ပြီး အထက်က python code ကိုပဲ update လုပ်ပြီး run ခဲ့တယ်။  

In [21]:
%pwd

'/mnt/disk1/ye/exp/LM-Tutorial'

In [30]:
import os
# This updates the PATH for the duration of the notebook session
os.environ['PATH'] = "/home/ye/tool/SRILM/bin/i686-m64:" + os.environ['PATH']

# Verify
!echo $PATH

/home/ye/tool/SRILM/bin/i686-m64:/home/ye/tool/SRILM/bin/i686-m64:/home/ye/tool/speech_tools/bin:/home/ye/tool/festival/bin:/home/ye/tool/festvox/bin:/home/ye/tool/flite/bin:/home/ye/.nvm/versions/node/v20.19.3/bin:/usr/lib/nvidia-cuda-toolkit/bin:/usr/bin:/bin:/home/ye/tool/marian/build:/home/ye/.local/bin:/home/ye/tool/speech_tools/bin:/home/ye/tool/festival/bin:/home/ye/tool/festvox/bin:/home/ye/tool/flite/bin:/home/ye/.nvm/versions/node/v20.19.3/bin:/home/ye/.cargo/bin:/usr/lib/nvidia-cuda-toolkit/bin:/usr/bin:/bin:/home/ye/tool/marian/build:/home/ye/miniforge3/condabin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin:/home/ye/.local/bin:/home/ye/tool/kenlm/build/bin/:/home/ye/.local/bin:/home/ye/.local/bin:/home/ye/tool/kenlm/build/bin/


In [31]:
# Basic 5-gram with Kneser-Ney smoothing (recommended)
!ngram-count \
    -order 5 \
    -text ./data/mypos_v3.word.clean \
    -lm ./srilm/myanmar_srilm.arpa \
    -kndiscount \
    -interpolate \
    -unk \
    -map-unk "<unk>"

### Flag explanations:

-**kndiscount**: Kneser-Ney smoothing (best general-purpose smoothing, comparable to KenLM's default)  
-**interpolate**: interpolates higher and lower order n-grams (standard practice)  
-**unk / -map-unk**: handles out-of-vocabulary words  

In [32]:
!ls ./srilm/

myanmar_srilm.arpa


## Convert ARPA to KenLM Binary (Optional but faster loading)

In [33]:
!build_binary ./srilm/myanmar_srilm.arpa ./srilm/myanmar_srilm.binary

Reading ./srilm/myanmar_srilm.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
SUCCESS


In [34]:
!ls ./srilm

myanmar_srilm.arpa  myanmar_srilm.binary


In [37]:
!ls -lh ./srilm/*

-rw-rw-r-- 1 ye ye  14M May  2 20:35 ./srilm/myanmar_srilm.arpa
-rw-r--r-- 1 ye ye 7.0M May  2 20:38 ./srilm/myanmar_srilm.binary


In [38]:
# Quick PPL check using SRILM's own tool
!ngram \
    -order 5 \
    -lm ./srilm/myanmar_srilm.arpa \
    -ppl ./data/otest.word.clean \
    -unk

file ./data/otest.word.clean: 1000 sentences, 12204 words, 0 OOVs
0 zeroprobs, logprob= -21133.62 ppl= 39.86085 ppl1= 53.91329


## Updated Code

အထက်က ဆောက်ထားတဲ့ SRILM မော်ဒယ်ကိုထပ်ဖြည့်ထားတာပါ။  
ပိုမြန်အောင် Binary LM ကိုပဲ သုံးကြရအောင်။ 

In [40]:
import math
import json
import pandas as pd
import torch
from pathlib import Path
import sys

sys.path.append(str(Path.cwd() / "lstm"))
sys.path.append(str(Path.cwd() / "transformer"))

import kenlm
from lstm_lm import TextDataset as LSTM_Dataset, LSTM_LM as LSTM_Model, Vocabulary
from transformer_lm import evaluate_ppl as transformer_evaluate_ppl

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

TEST_FILE = "./data/otest.word.clean"

# ==========================================
# SHARED METRIC FUNCTION (reused for both
# KenLM and SRILM since both produce ARPA)
# ==========================================
def get_ngram_metrics(path, model):
    """
    Works for any kenlm-loadable model (KenLM binary or SRILM ARPA).
    Returns (ppl, entropy_nats, bpc).
    """
    sum_log10, n_words, n_chars = 0.0, 0, 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            for prob, length, oov in model.full_scores(line):
                sum_log10 += prob
                n_words += 1        # count </s> token too
            n_chars += len(line)

    sum_nats   = -sum_log10 * math.log(10)
    total_bits = -sum_log10 * math.log2(10)

    ppl = math.exp(sum_nats / n_words)  if n_words else float("inf")
    ent = sum_nats / n_words            if n_words else float("inf")
    bpc = total_bits / n_chars          if n_chars else float("inf")

    return ppl, ent, bpc

# ==========================================
# 1. EVALUATE KENLM (N-gram)
# ==========================================
print("Evaluating KenLM...")
model_kenlm = kenlm.LanguageModel("./kenlm/myanmar_lm.binary")
ppl_kenlm, ent_kenlm, bpc_kenlm = get_ngram_metrics(TEST_FILE, model_kenlm)

# ==========================================
# 2. EVALUATE SRILM (N-gram)
# ==========================================
print("Evaluating SRILM...")
# Load ARPA directly — or use .binary if you ran build_binary
model_srilm = kenlm.LanguageModel("./srilm/myanmar_srilm.binary")
ppl_srilm, ent_srilm, bpc_srilm = get_ngram_metrics(TEST_FILE, model_srilm)

# ==========================================
# 3. EVALUATE LSTM (Word-level)
# ==========================================
print("Evaluating LSTM...")
with open("./lstm/myanmar_word_lm.pt.vocab", "r", encoding="utf-8") as f:
    token2idx = json.load(f)
vocab_lstm = Vocabulary()
vocab_lstm.token2idx = token2idx
vocab_lstm.idx2token = {int(v): k for k, v in token2idx.items()}

model_lstm = LSTM_Model(len(vocab_lstm)).to(device)
model_lstm.load_state_dict(torch.load("./lstm/myanmar_word_lm.pt", map_location=device))
model_lstm.eval()

with open(TEST_FILE, 'r', encoding='utf-8') as f:
    test_tokens = f.read().split()

test_ds   = LSTM_Dataset(test_tokens, vocab_lstm, seq_len=50)
criterion = torch.nn.CrossEntropyLoss()
nll_sum, n_tokens = 0.0, 0

with torch.no_grad():
    for i in range(len(test_ds)):
        x, y = test_ds[i]
        x = x.unsqueeze(0).to(device)
        y = y.unsqueeze(0).to(device)
        out, _ = model_lstm(x)
        loss = criterion(out.view(-1, out.size(-1)), y.view(-1))
        nll_sum  += loss.item() * y.numel()
        n_tokens += y.numel()

ppl_lstm = math.exp(nll_sum / n_tokens)
ent_lstm = nll_sum / n_tokens

# ==========================================
# 4. EVALUATE TRANSFORMER (XGLM)
# ==========================================
print("Evaluating XGLM...")
class Args:
    pass
args = Args()
args.seq_len = 128
args.stride  = 64

from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer_xglm = AutoTokenizer.from_pretrained("./transformer/myanmar_xglm_word")
model_xglm = AutoModelForCausalLM.from_pretrained(
    "./transformer/myanmar_xglm_word",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager"
)

with open(TEST_FILE, 'r', encoding='utf-8') as f:
    test_lines_xglm = [line.strip() for line in f if line.strip()]

ppl_xglm, nll_xglm = transformer_evaluate_ppl(args, model_xglm, tokenizer_xglm, test_lines_xglm)
ent_xglm = nll_xglm

# ==========================================
# 5. COMPILE THE RESULTS TABLE
# ==========================================
results = [
    {
        "Model":                  "KenLM (Myanmar 5-gram)",
        "Test PPL":               f"{ppl_kenlm:.2f}",
        "Test Entropy (nats)":    f"{ent_kenlm:.2f}",
        "Test BPC":               f"{bpc_kenlm:.3f}"
    },
    {
        "Model":                  "SRILM (Myanmar 5-gram)",
        "Test PPL":               f"{ppl_srilm:.2f}",
        "Test Entropy (nats)":    f"{ent_srilm:.2f}",
        "Test BPC":               f"{bpc_srilm:.3f}"
    },
    {
        "Model":                  "PyTorch LSTM (GPU)",
        "Test PPL":               f"{ppl_lstm:.2f}",
        "Test Entropy (nats)":    f"{ent_lstm:.2f}",
        "Test BPC":               "N/A (Word level)"
    },
    {
        "Model":                  "XGLM (Transformer, GPU)",
        "Test PPL":               f"{ppl_xglm:.2f}",
        "Test Entropy (nats)":    f"{ent_xglm:.2f}",
        "Test BPC":               "N/A (Subword level)"
    }
]

df = pd.DataFrame(results)
print(df)

Using device: cuda
Evaluating KenLM...
Evaluating SRILM...
Evaluating LSTM...
Evaluating XGLM...
Tokenizing test set (1000 lines)...
Evaluating with Stride=64, Context=128...
                     Model Test PPL Test Entropy (nats)             Test BPC
0   KenLM (Myanmar 5-gram)     7.70                2.04                0.586
1   SRILM (Myanmar 5-gram)    39.86                3.69                1.058
2       PyTorch LSTM (GPU)     2.12                0.75     N/A (Word level)
3  XGLM (Transformer, GPU)    18.21                2.90  N/A (Subword level)


## Summary

အခုချိန်မှာတော့ traditional လို့ ပြောရမယ့် Statistical LM ကနေ၊ Neural Network based LM တွေ (LSTM, Transformer) နဲ့ Large Language Model တွေနဲ့တွဲသုံးတဲ့ RAG  အထိပါ ဒေတာပြင်တာကနေ မော်ဒယ်ဆောက်တဲ့အပိုင်း၊ ပြီးတော့ ဆောက်ထားတဲ့ မော်ဒယ်တွေကနေ ဘယ်လို text generation လုပ်လို့ ရသလဲ စသည်ဖြင့် အဆင့်ဆင့် လက်တွေ့လုပ်ပြခဲ့ပါတယ်။ နောက်ဆုံး ဆောက်ထားတာမတူညီကြတဲ့ LM တွေအကြား မှာ Bits-Per-Character (BPC) နဲ့ ပြန်ညှိပြီး evaluation လုပ်တာကိုလည်း သင်ကြားပေးခဲ့ပါတယ်။  

ဆရာ ပြောရဲပါတယ်။ LM နဲ့ပတ်သက်ပြီး ဒီလောက်စုံအောင် လက်တွေ့လုပ်ပြထားတဲ့ Notebook က ရှားပါလိမ့်မယ်။ လုပ်ပြထားတဲ့ အဆင့်တွေအားလုံးကို ဒေတာကိုပြောင်းလိုက်ပြီး ကိုယ်တိုင် အစအဆုံး ပြီးအောင် run ကြည့်ရင်းနဲ့ ကြိုးစားလေ့လာသွားကြပါလို့။  

## Reference Links

- SRILM Homepage: [http://www.speech.sri.com/projects/srilm/](http://www.speech.sri.com/projects/srilm/)
- SRILM GitHub: [https://github.com/BitSpeech/SRILM](https://github.com/BitSpeech/SRILM)
- SRILM Installation Note: [https://github.com/ye-kyaw-thu/error-overflow/blob/master/SRILM-installation-note.md](https://github.com/ye-kyaw-thu/error-overflow/blob/master/SRILM-installation-note.md)
- IRSTLM: [https://github.com/irstlm-team/irstlm](https://github.com/irstlm-team/irstlm)
- KenLM: [https://github.com/kpu/kenlm](https://github.com/kpu/kenlm)
- KenLM Paper: [https://kheafield.com/papers/avenue/kenlm.pdf](https://kheafield.com/papers/avenue/kenlm.pdf)
- Training an n-gram Language Model and Estimating Sentence Probability by Dr. Masato Hagiwara: [https://masatohagiwara.net/training-an-n-gram-language-model-and-estimating-sentence-probability.html](https://masatohagiwara.net/training-an-n-gram-language-model-and-estimating-sentence-probability.html)
- myPOS Corpus Version 3 (ဒီ Tutorial အတွက် သုံးခဲ့တဲ့ monolingual corpus ပါ): [https://github.com/ye-kyaw-thu/myPOS](https://github.com/ye-kyaw-thu/myPOS)  